
# 01. Dataset Audit & Data Quality

## Objective
This notebook explores the Olist Brazilian E-Commerce dataset to understand its structure, identify data-quality issues, and assess its suitability for business analysis.

## Scope
- Inspect table dimensions, column names, and data types.
- Identify missing values and duplicate records.
- Understand table grain, primary keys, and relationships.
- Document data-quality issues before cleaning and transformation.

## Business Context
The project investigates marketplace sales, customer retention, product performance, and delivery operations. Reliable analysis depends on understanding what each record represents and whether the underlying data is complete and consistent.

## Expected Outcome
A documented dataset audit, validated key assumptions, and a foundation for data cleaning, SQL analysis, and business intelligence.

**Dataset:** Olist Brazilian E-Commerce Public Dataset

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

for file in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file)

    print(f"\n{'=' * 60}")
    print(f"FILE: {file.name}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print("\nColumn names:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)


FILE: olist_customers_dataset.csv
Rows: 99,441
Columns: 5

Column names:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

FILE: olist_geolocation_dataset.csv
Rows: 1,000,163
Columns: 5

Column names:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

FILE: olist_order_items_dataset.csv
Rows: 112,650
Columns: 7

Column names:
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Data types:
order_id            

In [3]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

audit_results = []

for file in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file)

    audit_results.append({
        "table": file.stem,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": df.isna().sum().sum(),
        "missing_percent": round(
            df.isna().sum().sum() / df.size * 100, 2
        ),
        "duplicate_rows": df.duplicated().sum()
    })

audit_df = pd.DataFrame(audit_results)

audit_df

,table,rows,columns,missing_cells,missing_percent,duplicate_rows
0,olist_customers_dataset,99441,5,0,0.00,0
1,olist_geolocation_dataset,1000163,5,0,0.00,261831
2,olist_order_items_dataset,112650,7,0,0.00,0
3,olist_order_payments_dataset,103886,5,0,0.00,0
4,olist_order_reviews_dataset,99224,7,145903,21.01,0
5,olist_orders_dataset,99441,8,4908,0.62,0
6,olist_products_dataset,32951,9,2448,0.83,0
7,olist_sellers_dataset,3095,4,0,0.00,0
8,product_category_name_translation,71,2,0,0.00,0


In [4]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

for file in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file)

    missing = df.isna().sum()
    missing = missing[missing > 0]

    if not missing.empty:
        print(f"\n{'=' * 55}")
        print(file.name)

        result = pd.DataFrame({
            "missing_count": missing,
            "missing_percent": (
                missing / len(df) * 100
            ).round(2)
        })

        print(result)


olist_order_reviews_dataset.csv
                        missing_count  missing_percent
review_comment_title            87656            88.34
review_comment_message          58247            58.70

olist_orders_dataset.csv
                               missing_count  missing_percent
order_approved_at                        160             0.16
order_delivered_carrier_date            1783             1.79
order_delivered_customer_date           2965             2.98

olist_products_dataset.csv
                            missing_count  missing_percent
product_category_name                 610             1.85
product_name_lenght                   610             1.85
product_description_lenght            610             1.85
product_photos_qty                    610             1.85
product_weight_g                        2             0.01
product_length_cm                       2             0.01
product_height_cm                       2             0.01
product_width_cm            


## Preliminary Findings: Missing Values

The initial audit identified missing values in the reviews, orders, and products tables.

- **Reviews:** Missing review titles and messages are common. These fields require separate investigation from the review score.
- **Orders:** Missing approval and delivery timestamps may be associated with order status and fulfillment progress.
- **Products:** Missing category and descriptive attributes may limit category-level and product-level analysis. Missing physical dimensions may affect shipping analysis.

Missing values will not be removed or imputed automatically. Their treatment will depend on business context and the intended analysis.

In [5]:
key_checks = {
    "customers": (
        "olist_customers_dataset.csv",
        ["customer_id"]
    ),
    "orders": (
        "olist_orders_dataset.csv",
        ["order_id"]
    ),
    "products": (
        "olist_products_dataset.csv",
        ["product_id"]
    ),
    "sellers": (
        "olist_sellers_dataset.csv",
        ["seller_id"]
    ),
    "order_items": (
        "olist_order_items_dataset.csv",
        ["order_id", "order_item_id"]
    ),
    "payments": (
        "olist_order_payments_dataset.csv",
        ["order_id", "payment_sequential"]
    ),
    "reviews": (
        "olist_order_reviews_dataset.csv",
        ["review_id"]
    ),
    "category_translation": (
        "product_category_name_translation.csv",
        ["product_category_name"]
    )
}

for table, (filename, key_columns) in key_checks.items():
    df = pd.read_csv(DATA_DIR / filename)

    duplicate_keys = df.duplicated(
        subset=key_columns
    ).sum()

    missing_keys = df[key_columns].isna().any(axis=1).sum()

    print(
        f"{table:24} | "
        f"Duplicate keys: {duplicate_keys:5} | "
        f"Missing keys: {missing_keys:5}"
    )

customers                | Duplicate keys:     0 | Missing keys:     0
orders                   | Duplicate keys:     0 | Missing keys:     0
products                 | Duplicate keys:     0 | Missing keys:     0
sellers                  | Duplicate keys:     0 | Missing keys:     0
order_items              | Duplicate keys:     0 | Missing keys:     0
payments                 | Duplicate keys:     0 | Missing keys:     0
reviews                  | Duplicate keys:   814 | Missing keys:     0
category_translation     | Duplicate keys:     0 | Missing keys:     0


In [6]:
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")

print(reviews.shape)

(99224, 7)


In [7]:
duplicate_reviews = reviews[reviews.duplicated(subset = ["review_id"] , keep = False)].sort_values("review_id")

duplicate_reviews.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [8]:
duplicate_reviews["review_id"].value_counts().head(10)

review_id
08528f70f579f0c830189efc523d2182    3
0c76e7a547a531e7bf9f0b99cba071c1    3
1fb4ddc969e6bea80e38deec00393a6f    3
2172867fd5b1a55f98fe4608e1547b4b    3
2d6ac45f859465b5c185274a1c929637    3
308316408775d1600dad81bd3184556d    3
32415bbf6e341d5d517080a796f79b5c    3
3415c9f764e478409e8e0660ae816dd2    3
38821b5c496b678cf91acc34892805ad    3
39b4603793c1c7f5f36d809b4a218664    3
Name: count, dtype: int64

In [9]:
print("Exact duplicat rows : " , reviews.duplicated().sum() )

print("Unique review ID's :" , reviews["review_id"].nunique() )

print("Unique Order ID's :" , reviews["order_id"].nunique() )

Exact duplicat rows :  0
Unique review ID's : 98410
Unique Order ID's : 98673


In [10]:
review_check = (
    duplicate_reviews
    .groupby("review_id")
    .agg(
        row_count=("order_id", "size"),
        unique_orders=("order_id", "nunique"),
        unique_scores=("review_score", "nunique"),
        unique_messages=("review_comment_message", "nunique"),
        unique_creation_dates=("review_creation_date", "nunique"),
        unique_answer_dates=("review_answer_timestamp", "nunique")
    )
)

review_check.head(10)

,row_count,unique_orders,unique_scores,unique_messages,unique_creation_dates,unique_answer_dates
review_id,,,,,,
00130cbe1f9d422698c812ed8ded1919,2,2,1,1,1,1
0115633a9c298b6a98bcbe4eee75345f,2,2,1,0,1,1
0174caf0ee5964646040cd94e15ac95e,2,2,1,1,1,1
017808d29fd1f942d97e50184dfb4c13,2,2,1,0,1,1
0254bd905dc677a6078990aad3331a36,2,2,1,1,1,1
0288d42bef3dfe36930740c9588a570f,2,2,1,0,1,1
02aa7f5f75e964e3c7efa59a1f515281,2,2,1,0,1,1
034528cb00b6bc981847acafbdf2ae0b,2,2,1,0,1,1
03a6a25db577d0689440933055111897,2,2,1,1,1,1


In [11]:
review_check.describe()

,row_count,unique_orders,unique_scores,unique_messages,unique_creation_dates,unique_answer_dates
count,789.000000,789.000000,789.0,789.000000,789.0,789.0
mean,2.031686,2.031686,1.0,0.378961,1.0,1.0
std,0.175273,0.175273,0.0,0.485436,0.0,0.0
min,2.000000,2.000000,1.0,0.000000,1.0,1.0
25%,2.000000,2.000000,1.0,0.000000,1.0,1.0
50%,2.000000,2.000000,1.0,0.000000,1.0,1.0
75%,2.000000,2.000000,1.0,1.000000,1.0,1.0
max,3.000000,3.000000,1.0,1.000000,1.0,1.0


In [12]:
review_check[
    (review_check["unique_scores"] > 1) |
    (review_check["unique_messages"] > 1) |
    (review_check["unique_creation_dates"] > 1) |
    (review_check["unique_answer_dates"] > 1)
]

,row_count,unique_orders,unique_scores,unique_messages,unique_creation_dates,unique_answer_dates
review_id,,,,,,


In [13]:
print("Repeated order IDs:", reviews["order_id"].duplicated().sum())

print(
    "Review titles with conflicting values:",
    duplicate_reviews.groupby("review_id")["review_comment_title"]
    .nunique()
    .gt(1)
    .sum()
)

Repeated order IDs: 551
Review titles with conflicting values: 0


### Review table findings

The review table contains repeated review IDs, but no exact duplicate rows. Repeated review IDs are associated with multiple orders and have consistent scores, comments, titles, and timestamps across the checked records.

Some order IDs also appear in multiple review rows. These relationships require further investigation before deciding whether any records should be removed. The raw review data will remain unchanged for now.


### Referential Integrity Checks ###

Check whether foreign keys in each table match valid primary keys
in the related table.

First, validate the relationship between orders and customers.

In [14]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")


In [15]:
missing_customers = orders.loc[
    ~orders["customer_id"].isin(customers["customer_id"]),
    "customer_id"
]

print("Orders with missing customer records:", len(missing_customers))

Orders with missing customer records: 0


**Finding:** All orders reference valid customer records.
No referential integrity issues were found between the orders
and customers tables.

In [16]:
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")

missing_orders = order_items.loc[
    ~order_items["order_id"].isin(orders["order_id"]),
    "order_id"
]

print("Order items with missing orders:", len(missing_orders))

Order items with missing orders: 0


**Finding:** All order items reference valid orders.
No referential integrity issues were found between the
order_items and orders tables.

In [17]:
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")

missing_products = order_items.loc[
    ~order_items["product_id"].isin(products["product_id"]),
    "product_id"
]

print("Order items with missing products:", len(missing_products))


Order items with missing products: 0


**Finding:** All order items reference valid products.
No referential integrity issues were found between the
order_items and products tables.

In [18]:
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

missing_sellers = order_items.loc[
    ~order_items["seller_id"].isin(sellers["seller_id"]),
    "seller_id"
]

print("Order items with missing sellers:", len(missing_sellers))

Order items with missing sellers: 0


**Finding:** All order items reference valid sellers.
No referential integrity issues were found between the
order_items and sellers tables.

In [19]:
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")

missing_orders_payments = payments.loc[
    ~payments["order_id"].isin(orders["order_id"]),
    "order_id"
]

print("Payments with missing orders:", len(missing_orders_payments))

Payments with missing orders: 0


**Finding:** All payment records reference valid orders.
No referential integrity issues were found between the
order_payments and orders tables.

In [20]:
missing_review_orders = reviews.loc[
    ~reviews["order_id"].isin(orders["order_id"]),
    "order_id"
]

print("Reviews with missing orders:", len(missing_review_orders))

Reviews with missing orders: 0


**Finding:** All review records reference valid orders.
No referential integrity issues were found between the
order_reviews and orders tables.

In [21]:
translations = pd.read_csv(
    DATA_DIR / "product_category_name_translation.csv"
)

missing_translations = products.loc[
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(
        translations["product_category_name"]
    ),
    "product_category_name"
]

print("Product categories without translations:",
      missing_translations.nunique())

Product categories without translations: 2


In [22]:
missing_category_names = (
    missing_translations
    .dropna()
    .unique()
)

print("Missing categories:")
print(missing_category_names)

Missing categories:
<StringArray>
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 2, dtype: str


In [23]:
products[
    products["product_category_name"].isin(
        missing_category_names
    )
]["product_category_name"].value_counts()

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64

**Finding:** Two product categories are missing from the
English category translation table, affecting 13 products.

- `pc_gamer`: 3 products
- `portateis_cozinha_e_preparadores_de_alimentos`: 10 products

These categories will be preserved in the raw data. Translation
handling will be addressed during the data cleaning phase.

## Dataset Audit Summary

### Structural Checks
- Inspected all 9 CSV files for dimensions, missing values, and duplicate rows.
- Validated primary keys and identified repeated review IDs.
- Confirmed that order_items uses a composite key: order_id + order_item_id.

### Referential Integrity
- Orders reference valid customers.
- Order items reference valid orders, products, and sellers.
- Payments and reviews reference valid orders.
- Two product categories are missing from the English translation table, affecting 13 products.

### Data Quality Issues to Address
- Missing values in orders, products, and reviews.
- 261,831 exact duplicate rows in geolocation.
- Repeated review IDs and repeated order IDs in the review table.
- Missing English translations for two product categories.

### Audit Conclusion
The dataset structure and major table relationships have been examined.
Identified data quality issues will be investigated and handled during
the data cleaning phase. The original CSV files remain unchanged.

## Phase 2: Data Cleaning

### 2.1 Orders Table: Missing Values

Inspect missing values in the orders table to understand
which order lifecycle timestamps are incomplete and how
this may affect delivery performance analysis.

In [24]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [25]:
missing_orders = pd.DataFrame({
    "missing_count": orders.isnull().sum(),
    "missing_percent": (
        orders.isnull().mean() * 100
    ).round(2)
})

missing_orders[missing_orders["missing_count"] > 0]

,missing_count,missing_percent
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


In [26]:
orders.groupby("order_status").agg(
    total_orders=("order_id", "count"),
    missing_approved=("order_approved_at", "count"),
    missing_carrier=("order_delivered_carrier_date", "count"),
    missing_customer_delivery=("order_delivered_customer_date", "count")
)

,total_orders,missing_approved,missing_carrier,missing_customer_delivery
order_status,,,,
approved,2,2,0,0
canceled,625,484,75,6
created,5,0,0,0
delivered,96478,96464,96476,96470
invoiced,314,314,0,0
processing,301,301,0,0
shipped,1107,1107,1107,0
unavailable,609,609,0,0


In [27]:
status_missing = orders.groupby("order_status").agg(
    total_orders=("order_id", "count"),
    missing_approved=("order_approved_at", lambda x: x.isna().sum()),
    missing_carrier=("order_delivered_carrier_date", lambda x: x.isna().sum()),
    missing_customer_delivery=("order_delivered_customer_date", lambda x: x.isna().sum())
)

status_missing

,total_orders,missing_approved,missing_carrier,missing_customer_delivery
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [28]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_approved_at"].isna())
][[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00
26800,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaN,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01 00:00:00
38290,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaN,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27 00:00:00
39334,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaN,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22 00:00:00
48401,7002a78c79c519ac54022d4f8a65e6e8,delivered,2017-01-19 22:26:59,NaN,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16 00:00:00
61743,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaN,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20 00:00:00


In [29]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
][[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [30]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [31]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

**Finding:** Eight orders have `order_status = delivered` but are
missing `order_delivered_customer_date`. Seven have a recorded carrier
handoff date, while one is also missing the carrier timestamp.

These records will be retained without imputation. Delivery-time analyses
will exclude orders without a recorded customer delivery timestamp.

In [32]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [33]:
orders[timestamp_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [34]:
orders[timestamp_cols].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

***Timestamp Standardization***

The order lifecycle timestamp columns were converted from string
values to datetime format using `pd.to_datetime()`.

Missing-value counts remained unchanged after conversion, confirming
that the transformation did not introduce additional missing values.

Missing timestamps will be retained where they represent incomplete
order lifecycle stages. No timestamps will be artificially imputed.

In [35]:
invalid_order_dates = orders[
    (orders["order_approved_at"] < orders["order_purchase_timestamp"]) |
    (orders["order_delivered_carrier_date"] < orders["order_approved_at"]) |
    (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
]

print("Orders with invalid timestamp sequence:", len(invalid_order_dates))

Orders with invalid timestamp sequence: 1382


In [36]:
timestamp_issues = {
    "approval_before_purchase": (
        orders["order_approved_at"].notna() &
        (orders["order_approved_at"] < orders["order_purchase_timestamp"])
    ),

    "carrier_before_approval": (
        orders["order_approved_at"].notna() &
        orders["order_delivered_carrier_date"].notna() &
        (orders["order_delivered_carrier_date"] < orders["order_approved_at"])
    ),

    "delivery_before_carrier": (
        orders["order_delivered_carrier_date"].notna() &
        orders["order_delivered_customer_date"].notna() &
        (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"])
    )
}

for issue, condition in timestamp_issues.items():
    print(f"{issue}: {condition.sum()}")

approval_before_purchase: 0
carrier_before_approval: 1359
delivery_before_carrier: 23


In [37]:
carrier_before_approval = timestamp_issues["carrier_before_approval"]

orders.loc[
    carrier_before_approval,
    "order_status"
].value_counts()

order_status
delivered    1350
shipped         9
Name: count, dtype: int64

In [38]:
time_gap = (
    orders.loc[carrier_before_approval, "order_delivered_carrier_date"]
    - orders.loc[carrier_before_approval, "order_approved_at"]
)

time_gap.describe()

count                        1359
mean     -2 days +23:14:52.846947
std        4 days 19:18:28.055754
min           -172 days +18:44:38
25%             -2 days +22:02:36
50%             -1 days +06:49:56
75%      -1 days +22:35:04.500000
max             -1 days +23:59:39
dtype: object

In [39]:
orders.loc[
    carrier_before_approval
].assign(
    approval_to_carrier=time_gap
).sort_values(
    "approval_to_carrier"
).head(10)[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "approval_to_carrier"
]]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,approval_to_carrier
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,-172 days +18:44:38
14562,1fab4ac9d85079b3da72a11475ae1685,delivered,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,2017-09-08 20:13:03,-10 days +15:04:12
46163,0184d4ddb259e1a4cfc2871888cf97b8,delivered,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,2017-09-09 15:12:44,-10 days +15:48:35
98710,1378f9601350615613cc8832d6789c5d,delivered,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,2017-09-13 22:24:46,-10 days +20:04:04
41592,8554cb37f7158cb0b082a841d24a4589,delivered,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,2017-09-08 20:07:45,-10 days +21:14:15
11738,cf72398d0690f841271b695bbfda82d2,delivered,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-04 20:12:41,2017-09-11 14:15:02,-10 days +22:08:02
85393,40de47dfa620d667117e4a6067b6e1ec,delivered,2017-09-01 20:05:55,2017-09-13 21:58:38,2017-09-04 20:36:58,2017-09-08 20:41:16,-10 days +22:38:20
31211,bc4854efd86d9f42140c951c595d20c1,delivered,2017-09-01 20:05:42,2017-09-13 22:00:51,2017-09-04 20:49:57,2017-09-06 22:15:45,-10 days +22:49:06
55302,77ca435b03fbf991e5027e3776e37885,delivered,2017-09-01 18:49:54,2017-09-13 21:56:08,2017-09-04 20:50:00,2017-09-30 17:33:43,-10 days +22:53:52
68315,580f3268bc2075c5961b2d2929c7a35b,delivered,2017-09-01 20:17:09,2017-09-13 22:03:42,2017-09-05 11:58:18,2017-09-18 17:52:20,-9 days +13:54:36


In [40]:
orders.loc[
    carrier_before_approval
].assign(
    order_id=lambda x: x["order_id"].str[:8],
    approval_to_carrier=time_gap
).sort_values(
    "approval_to_carrier"
).head(10)[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "approval_to_carrier"
]]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,approval_to_carrier
25883,7c48bb55,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,-172 days +18:44:38
14562,1fab4ac9,delivered,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,2017-09-08 20:13:03,-10 days +15:04:12
46163,0184d4dd,delivered,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,2017-09-09 15:12:44,-10 days +15:48:35
98710,1378f960,delivered,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,2017-09-13 22:24:46,-10 days +20:04:04
41592,8554cb37,delivered,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,2017-09-08 20:07:45,-10 days +21:14:15
11738,cf72398d,delivered,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-04 20:12:41,2017-09-11 14:15:02,-10 days +22:08:02
85393,40de47df,delivered,2017-09-01 20:05:55,2017-09-13 21:58:38,2017-09-04 20:36:58,2017-09-08 20:41:16,-10 days +22:38:20
31211,bc4854ef,delivered,2017-09-01 20:05:42,2017-09-13 22:00:51,2017-09-04 20:49:57,2017-09-06 22:15:45,-10 days +22:49:06
55302,77ca435b,delivered,2017-09-01 18:49:54,2017-09-13 21:56:08,2017-09-04 20:50:00,2017-09-30 17:33:43,-10 days +22:53:52
68315,580f3268,delivered,2017-09-01 20:17:09,2017-09-13 22:03:42,2017-09-05 11:58:18,2017-09-18 17:52:20,-9 days +13:54:36


In [41]:
approval_delay_days = (
    orders.loc[carrier_before_approval, "order_approved_at"]
    - orders.loc[carrier_before_approval, "order_delivered_carrier_date"]
).dt.total_seconds() / (24 * 60 * 60)

pd.Series({
    "More than 1 day": (approval_delay_days > 1).sum(),
    "More than 7 days": (approval_delay_days > 7).sum(),
    "More than 30 days": (approval_delay_days > 30).sum(),
    "More than 90 days": (approval_delay_days > 90).sum()
})

More than 1 day      458
More than 7 days      14
More than 30 days      1
More than 90 days      1
dtype: int64

In [42]:
orders.loc[
    carrier_before_approval,
    "order_approved_at"
].dt.date.value_counts().head(15)

order_approved_at
2018-07-05    451
2018-04-24    383
2018-08-01     22
2018-07-26     18
2018-07-23     18
2018-07-27     18
2018-02-04     18
2017-08-02     18
2018-06-27     15
2018-07-28     14
2018-08-20     13
2017-04-27     13
2017-09-13     12
2018-06-12     11
2017-04-28     10
Name: count, dtype: int64

In [43]:
approval_delay = (
    orders.loc[carrier_before_approval, "order_approved_at"]
    - orders.loc[carrier_before_approval, "order_purchase_timestamp"]
)

approval_delay.describe()

count                      1359
mean     2 days 06:31:10.913907
std      2 days 01:02:31.242442
min             0 days 00:07:28
25%             0 days 20:37:47
50%             1 days 22:13:58
75%      3 days 06:29:21.500000
max            12 days 03:24:38
dtype: object

In [44]:
delivery_before_carrier = timestamp_issues["delivery_before_carrier"]

orders.loc[
    delivery_before_carrier
].assign(
    order_id=lambda x: x["order_id"].str[:8]
)[[
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb65,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d4679,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01d,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11ca,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e3758,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


Timestamp Sequence Anomalies

A chronological validation identified 1,382 orders with timestamp
sequence issues.

- 1,359 orders have carrier timestamps earlier than approval timestamps.
- 23 orders have customer delivery timestamps earlier than carrier
  timestamps.
- No orders have approval timestamps earlier than purchase timestamps.

The timestamp inconsistencies are retained rather than corrected because
the source data does not provide enough information to determine which
timestamp is incorrect.

Records with invalid timestamp sequences will be excluded from analyses
that depend on the affected time intervals, while remaining available
for analyses that do not depend on those timestamps.

In [45]:
orders["invalid_timestamp_sequence"] = (
    timestamp_issues["approval_before_purchase"] |
    timestamp_issues["carrier_before_approval"] |
    timestamp_issues["delivery_before_carrier"]
)

orders["invalid_timestamp_sequence"].value_counts()

invalid_timestamp_sequence
False    98059
True      1382
Name: count, dtype: int64

In [46]:
orders["carrier_before_approval"] = timestamp_issues[
    "carrier_before_approval"
]

orders["delivery_before_carrier"] = timestamp_issues[
    "delivery_before_carrier"
]

In [47]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,invalid_timestamp_sequence,carrier_before_approval,delivery_before_carrier
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False,False,False


In [48]:
orders[
    [
        "invalid_timestamp_sequence",
        "carrier_before_approval",
        "delivery_before_carrier"
    ]
].sum()

invalid_timestamp_sequence    1382
carrier_before_approval       1359
delivery_before_carrier         23
dtype: int64

 Orders Table Cleaning Status

The order lifecycle timestamps were converted to datetime format and
validated for missing values and chronological consistency.

Missing timestamps were retained because they can represent incomplete
order stages. No timestamps were imputed.

Timestamp sequence validation identified 1,382 anomalous orders:
- 1,359 have carrier timestamps earlier than approval timestamps.
- 23 have customer delivery timestamps earlier than carrier timestamps.

The original timestamps are preserved, and explicit quality flags were
created to identify these records for downstream analysis.

Orders without valid timestamp sequences will be excluded only from
analyses that depend on the affected time intervals.

In [49]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [50]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [51]:
products[
    products["product_category_name"].isna()
][[
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].head(20)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [52]:
products[
    products["product_weight_g"].isna()
][[
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
products[
    products["product_weight_g"].isna()
]["product_id"].isin(order_items["product_id"]).value_counts()

product_id
True    2
Name: count, dtype: int64

In [54]:
products[
    products["product_category_name"].isna()
]["product_id"].isin(order_items["product_id"]).value_counts()

product_id
True    610
Name: count, dtype: int64

In [55]:
products[
    products["product_category_name"].notna() &
    ~products["product_category_name"].isin(
        translations["product_category_name"]
    )
][[
    "product_id",
    "product_category_name"
]]

,product_id,product_category_name
1628,0105b5323d24fc655f73052694dbbb3a,pc_gamer
5821,6fd83eb3e0799b775e4f946bd66657c0,portateis_cozinha_e_preparadores_de_alimentos
7325,5d923ead886c44b86845f69e50520c3e,portateis_cozinha_e_preparadores_de_alimentos
7478,6727051471a0fc4a0e7737b57bff2549,pc_gamer
8819,bed164d9d628cf0593003389c535c6e0,portateis_cozinha_e_preparadores_de_alimentos
11039,1220978a08a6b29a202bc015b18250e9,portateis_cozinha_e_preparadores_de_alimentos
14266,ae62bb0f95af63d64eae5f93dddea8d3,portateis_cozinha_e_preparadores_de_alimentos
16182,1954739d84629e7323a4295812a3e0ec,portateis_cozinha_e_preparadores_de_alimentos
16930,dbe520fb381ad695a7e1f2807d20c765,pc_gamer
17800,c7a3f1a7f9eef146cc499368b578b884,portateis_cozinha_e_preparadores_de_alimentos


In [56]:
products.dtypes

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [57]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [58]:
reviews[
    ["review_comment_title", "review_comment_message"]
].isna().value_counts()

review_comment_title  review_comment_message
True                  True                      56518
                      False                     31138
False                 False                      9839
                      True                       1729
Name: count, dtype: int64

In [59]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [60]:
reviews.dtypes

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [61]:
review_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in review_date_cols:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

In [62]:
reviews.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [63]:
invalid_review_dates = reviews[
    reviews["review_answer_timestamp"] < reviews["review_creation_date"]
]

print("Reviews with answer before creation:", len(invalid_review_dates))

Reviews with answer before creation: 0


In [64]:
review_id_counts = reviews["review_id"].value_counts()

review_id_counts.value_counts().sort_index()

count
1    97621
2      764
3       25
Name: count, dtype: int64

In [65]:
repeated_review_ids = review_id_counts[review_id_counts > 1].index

reviews[
    reviews["review_id"].isin(repeated_review_ids)
].groupby("review_id")["order_id"].nunique().value_counts().sort_index()

order_id
2    764
3     25
Name: count, dtype: int64

In [66]:
reviews[
    reviews["review_id"].isin(repeated_review_ids)
].groupby("review_id")["review_score"].nunique().value_counts().sort_index()

review_score
1    789
Name: count, dtype: int64

In [67]:
order_review_counts = reviews["order_id"].value_counts()

order_review_counts.value_counts().sort_index()

count
1    98126
2      543
3        4
Name: count, dtype: int64

In [68]:
multi_review_orders = order_review_counts[order_review_counts > 1].index

reviews[
    reviews["order_id"].isin(multi_review_orders)
].groupby("order_id")["review_score"].nunique().value_counts().sort_index()

review_score
1    345
2    202
Name: count, dtype: int64

In [69]:
geolocation = pd.read_csv(
    DATA_DIR / "olist_geolocation_dataset.csv"
)

In [70]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [71]:
geolocation["geolocation_zip_code_prefix"].value_counts().head(10)

geolocation_zip_code_prefix
24220    1146
24230    1102
38400     965
35500     907
11680     879
22631     832
30140     810
11740     788
38408     773
28970     743
Name: count, dtype: int64

In [72]:
print(geolocation.duplicated().sum())

261831


In [73]:
geolocation.drop_duplicates().shape

(738332, 5)

In [74]:
geolocation.value_counts().head()

geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  geolocation_city  geolocation_state
88220                        -27.102099       -48.629613       itapema           SC                   314
6414                         -23.495901       -46.874687       barueri           SP                   189
                             -23.490618       -46.869004       barueri           SP                   127
5145                         -23.506049       -46.717377       sao paulo         SP                   126
22620                        -23.005514       -43.375964       rio de janeiro    RJ                   102
Name: count, dtype: int64

In [75]:
geolocation[
    geolocation["geolocation_zip_code_prefix"] == 1046
].shape

(141, 5)

In [76]:
geolocation[
    geolocation["geolocation_zip_code_prefix"] == 1046
].drop_duplicates()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
19,1046,-23.545884,-46.643163,sao paulo,SP
20,1046,-23.545166,-46.643786,sao paulo,SP
33,1046,-23.545320,-46.644069,sao paulo,SP
57,1046,-23.546382,-46.642894,são paulo,SP
60,1046,-23.546397,-46.643059,sao paulo,SP
115,1046,-23.546253,-46.645217,sao paulo,SP
139,1046,-23.545318,-46.643975,sao paulo,SP
143,1046,-23.545369,-46.643183,sao paulo,SP


In [77]:
geolocation.groupby(
    ["geolocation_zip_code_prefix", "geolocation_city", "geolocation_state"]
).size().sort_values(ascending=False).head(10)

geolocation_zip_code_prefix  geolocation_city      geolocation_state
24220                        niteroi               RJ                   997
24230                        niteroi               RJ                   969
11680                        ubatuba               SP                   879
22631                        rio de janeiro        RJ                   832
30140                        belo horizonte        MG                   810
38400                        uberlandia            MG                   795
35500                        divinopolis           MG                   775
28970                        araruama              RJ                   743
36400                        conselheiro lafaiete  MG                   733
39400                        montes claros         MG                   724
dtype: int64

In [78]:
geolocation_clean = geolocation.drop_duplicates().copy()

In [79]:
geolocation_clean.shape

(738332, 5)

In [80]:
payments.dtypes

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

In [81]:
payments.columns

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')

In [82]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [83]:
payments[[
    "payment_sequential",
    "payment_installments",
    "payment_value"
]].describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [84]:
payments.groupby("payment_type")[
    ["payment_installments", "payment_value"]
].agg(
    count=("payment_value", "size"),
    min_installments=("payment_installments", "min"),
    max_installments=("payment_installments", "max"),
    total_value=("payment_value", "sum")
).sort_values("count", ascending=False)

,count,min_installments,max_installments,total_value
payment_type,,,,
credit_card,76795,0,24,12542084.19
boleto,19784,1,1,2869361.27
voucher,5775,1,1,379436.87
debit_card,1529,1,1,217989.79
not_defined,3,1,1,0.00


In [85]:
payments[
    (payments["payment_installments"] == 0) |
    (payments["payment_value"] == 0) |
    (payments["payment_type"] == "not_defined")
].sort_values(
    ["payment_type", "payment_installments", "payment_value"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00


In [86]:
suspicious_order_ids = payments.loc[
    (payments["payment_installments"] == 0) |
    (payments["payment_value"] == 0) |
    (payments["payment_type"] == "not_defined"),
    "order_id"
].unique()

payments[
    payments["order_id"].isin(suspicious_order_ids)
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00


In [87]:
payment_key_duplicates = payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

payment_sequence_check = payments.groupby("order_id").agg(
    payment_rows=("payment_sequential", "size"),
    unique_sequences=("payment_sequential", "nunique"),
    max_sequence=("payment_sequential", "max")
)

payment_sequence_check[
    payment_sequence_check["payment_rows"] !=
    payment_sequence_check["max_sequence"]
].head(10)

,payment_rows,unique_sequences,max_sequence
order_id,,,
00ac05fe0fc047c54418098eb64e3aaa,1,1,2
056c68d093c100017aab1f00f260705c,1,1,2
0668d086b3ae41adde3aed3dacbc8fae,1,1,2
0a7d898c6305101e69e9f5a05cd0130d,1,1,2
0d0e88a418636d40ea780339701db49c,1,1,2
0fdb6f73fab7005c84d4a43d8a93682b,1,1,2
159da9914b51de617fe80162939965c5,1,1,2
166b01d0fce75a595b92ad9e84e0843b,1,1,2
1896e8b8fd196151559b02e31e98ce89,1,1,2


In [88]:
# 1. Check duplicate composite keys
payment_key_duplicates

np.int64(0)

In [89]:
# 2. Count orders with sequence mismatches
sequence_mismatch_count = (
    payment_sequence_check["payment_rows"] !=
    payment_sequence_check["max_sequence"]
).sum()

sequence_mismatch_count

np.int64(80)

In [90]:
mismatched_orders = payment_sequence_check[
    payment_sequence_check["payment_rows"] !=
    payment_sequence_check["max_sequence"]
].index

payments[
    payments["order_id"].isin(mismatched_orders)
].sort_values(
    ["order_id", "payment_sequential"]
).head(30)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
27191,00ac05fe0fc047c54418098eb64e3aaa,2,debit_card,1,123.47
12785,056c68d093c100017aab1f00f260705c,2,debit_card,1,52.78
40318,0668d086b3ae41adde3aed3dacbc8fae,2,debit_card,1,252.74
84637,0a7d898c6305101e69e9f5a05cd0130d,2,credit_card,5,215.47
82233,0d0e88a418636d40ea780339701db49c,2,debit_card,1,121.09
62609,0fdb6f73fab7005c84d4a43d8a93682b,2,debit_card,1,34.13
88530,159da9914b51de617fe80162939965c5,2,debit_card,1,40.76
66314,166b01d0fce75a595b92ad9e84e0843b,2,credit_card,6,127.09
87490,1896e8b8fd196151559b02e31e98ce89,2,debit_card,1,108.95
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [91]:
payments[
    payments["order_id"].isin(mismatched_orders)
]["payment_sequential"].value_counts().sort_index()

payment_sequential
2    80
3     2
Name: count, dtype: int64

In [92]:
payment_sequence_check.loc[mismatched_orders].value_counts()

payment_rows  unique_sequences  max_sequence
1             1                 2               78
2             2                 3                2
Name: count, dtype: int64

In [93]:
payments[
    payments["order_id"].isin(mismatched_orders)
].sort_values(
    ["order_id", "payment_sequential"]
).groupby(
    "order_id"
).agg(
    payment_types=("payment_type", list),
    payment_values=("payment_value", list),
    sequences=("payment_sequential", list)
).head(15)

,payment_types,payment_values,sequences
order_id,,,
00ac05fe0fc047c54418098eb64e3aaa,[debit_card],[123.47],[2]
056c68d093c100017aab1f00f260705c,[debit_card],[52.78],[2]
0668d086b3ae41adde3aed3dacbc8fae,[debit_card],[252.74],[2]
0a7d898c6305101e69e9f5a05cd0130d,[credit_card],[215.47],[2]
0d0e88a418636d40ea780339701db49c,[debit_card],[121.09],[2]
0fdb6f73fab7005c84d4a43d8a93682b,[debit_card],[34.13],[2]
159da9914b51de617fe80162939965c5,[debit_card],[40.76],[2]
166b01d0fce75a595b92ad9e84e0843b,[credit_card],[127.09],[2]
1896e8b8fd196151559b02e31e98ce89,[debit_card],[108.95],[2]


In [94]:
payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [95]:
payments["payment_value"].agg(
    total="count",
    negative=lambda x: (x < 0).sum(),
    zero=lambda x: (x == 0).sum()
)

total       103886
negative         0
zero             9
Name: payment_value, dtype: int64

In [96]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [97]:
payments.groupby("payment_type").agg(
    min_installments=("payment_installments", "min"),
    max_installments=("payment_installments", "max"),
    zero_installments=(
        "payment_installments",
        lambda x: (x == 0).sum()
    )
)

,min_installments,max_installments,zero_installments
payment_type,,,
boleto,1,1,0
credit_card,0,24,2
debit_card,1,1,0
not_defined,1,1,0
voucher,1,1,0


Payments Table Audit

- **Shape:** 103,886 rows × 5 columns.
- **Grain:** One row per payment entry. Composite key: `order_id` + `payment_sequential`.
- **Missing values:** None.
- **Duplicate keys:** None.
- **Payment types:** Credit card is the most common (76,795 records), followed by boleto (19,784).
- **Payment values:** No negative values; 9 zero-value entries.
- **Installments:** Credit-card payments range from 0 to 24. Two entries have zero installments.
- **Sequence anomalies:** 80 orders have sequences that do not start at 1. 78 have only sequence 2, while 2 have sequences 2 and 3.
- **Unusual records:** Three `not_defined` entries have zero payment value. Zero-value voucher entries were also observed.

**Conclusion:** The table has no missing values or duplicate composite keys. Unusual payment values, installment counts, and sequence numbers were identified. No records were modified or removed during the audit.

In [98]:
order_items.shape

(112650, 7)

In [99]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [100]:
order_items.describe(include="all")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
count,112650,112650.000000,112650,112650,112650,112650.000000,112650.000000
unique,98666,NaN,32951,3095,93318,NaN,NaN
top,8272b63d03f5f79c56e9e4120aec44ef,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,6560211a19b47992c3666cc44a7e94c0,2018-03-01 02:50:48,NaN,NaN
freq,21,NaN,527,2033,21,NaN,NaN
mean,NaN,1.197834,NaN,NaN,NaN,120.653739,19.990320
std,NaN,0.705124,NaN,NaN,NaN,183.633928,15.806405
min,NaN,1.000000,NaN,NaN,NaN,0.850000,0.000000
25%,NaN,1.000000,NaN,NaN,NaN,39.900000,13.080000
50%,NaN,1.000000,NaN,NaN,NaN,74.990000,16.260000
75%,NaN,1.000000,NaN,NaN,NaN,134.900000,21.150000


In [101]:
order_items[[
    "price",
    "freight_value"
]].agg([
    "min",
    "max",
    "mean",
    "median"
])

,price,freight_value
min,0.850000,0.00000
max,6735.000000,409.68000
mean,120.653739,19.99032
median,74.990000,16.26000


In [102]:
(order_items[["price", "freight_value"]] < 0).sum()

price            0
freight_value    0
dtype: int64

In [103]:
(order_items[["price", "freight_value"]] == 0).sum()

price              0
freight_value    383
dtype: int64

In [104]:
order_items.nlargest(10, "price")[
    ["order_id", "product_id", "price", "freight_value"]
]

,order_id,product_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,3999.90,17.01


In [105]:
order_items.nlargest(10, "price")[
    ["order_id", "product_id", "price", "freight_value"]
]

,order_id,product_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,3999.90,17.01


In [106]:
order_items["freight_value"].eq(0).sum()

np.int64(383)

In [107]:
order_items.loc[
    order_items["freight_value"].eq(0),
    ["order_id", "product_id", "price", "freight_value"]
].head(10)

,order_id,product_id,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
258,00a870c6c06346e85335524935c600c0,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,53b36df67ebb7c41585e8d54d6772e08,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,53b36df67ebb7c41585e8d54d6772e08,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0


In [108]:
zero_freight = order_items[order_items["freight_value"].eq(0)]

{
    "zero_freight_rows": len(zero_freight),
    "percentage_of_items": round(len(zero_freight) / len(order_items) * 100, 2),
    "unique_orders": zero_freight["order_id"].nunique(),
    "unique_products": zero_freight["product_id"].nunique(),
    "unique_sellers": zero_freight["seller_id"].nunique()
}

{'zero_freight_rows': 383,
 'percentage_of_items': 0.34,
 'unique_orders': 339,
 'unique_products': 10,
 'unique_sellers': 9}

In [109]:
order_items["shipping_limit_date"].dtype

<StringDtype(storage='python', na_value=nan)>

In [110]:
order_items["shipping_limit_date"].head()

0    2017-09-19 09:45:35
1    2017-05-03 11:05:13
2    2018-01-18 14:48:30
3    2018-08-15 10:10:18
4    2017-02-13 13:57:51
Name: shipping_limit_date, dtype: str

In [111]:
shipping_dates = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

{
    "invalid_dates": shipping_dates.isna().sum(),
    "earliest_date": shipping_dates.min(),
    "latest_date": shipping_dates.max()
}

{'invalid_dates': np.int64(0),
 'earliest_date': Timestamp('2016-09-19 00:15:34'),
 'latest_date': Timestamp('2020-04-09 22:35:08')}

In [112]:
items_orders = order_items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left"
)

In [113]:
items_orders["order_purchase_timestamp"] = pd.to_datetime(
    items_orders["order_purchase_timestamp"],
    errors="coerce"
)

In [114]:
items_orders["shipping_limit_date"] = pd.to_datetime(
    items_orders["shipping_limit_date"],
    errors="coerce"
)

items_orders["shipping_limit_date"].lt(
    items_orders["order_purchase_timestamp"]
).sum()

np.int64(0)

In [115]:
shipping_window = (
    items_orders["shipping_limit_date"]
    - items_orders["order_purchase_timestamp"]
)

shipping_window.describe()

count                    112650
mean     6 days 15:27:03.728308
std      7 days 01:39:33.041016
min             2 days 00:06:06
25%             5 days 00:09:53
50%             6 days 00:17:16
75%             7 days 04:30:03
max          1056 days 01:00:00
dtype: object

In [116]:
items_orders.assign(
    shipping_window=shipping_window
).nlargest(
    10, "shipping_window"
)[[
    "order_id",
    "product_id",
    "seller_id",
    "order_purchase_timestamp",
    "shipping_limit_date",
    "shipping_window"
]]

,order_id,product_id,seller_id,order_purchase_timestamp,shipping_limit_date,shipping_window
8643,13bdf405f961a6deec817d817f5c6624,96ea060e41bdecc64e2de00b97068975,7a241947449cc45dbfda4f9d0798d9d0,2017-03-16 02:30:51,2020-02-05 03:30:51,1056 days 01:00:00
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,282b126b2354516c5f400154398f616d,7a241947449cc45dbfda4f9d0798d9d0,2017-03-14 19:23:22,2020-02-03 20:23:22,1056 days 01:00:00
85729,c2bb89b5c1dd978d507284be78a04cb2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2017-05-23 22:28:36,2020-04-09 22:35:08,1052 days 00:06:32
85730,c2bb89b5c1dd978d507284be78a04cb2,87b92e06b320e803d334ac23966c80b1,7a241947449cc45dbfda4f9d0798d9d0,2017-05-23 22:28:36,2020-04-09 22:35:08,1052 days 00:06:32
46557,69d126e78947276280838ee9361f5505,984a3b9f9bb4c8feb319da951212696e,88af55b4a7ca402b27df16f7c7c9b5d2,2017-03-30 15:23:23,2017-08-25 15:32:11,148 days 00:08:48
28492,40dc2ba6f322a17626aac6244332828c,38841dec90eddf2f43cdfabc48b3831a,1354d51653f645349064725ed204e85e,2017-10-05 21:39:05,2018-01-10 22:49:21,97 days 01:10:16
87724,c72727d29cde4cf870d569bf65edabfd,ee2f96cb63d491d963a67285f048775a,fce62094ffe6a4009188ec44e681dfdd,2017-02-07 18:01:15,2017-05-12 17:01:15,93 days 23:00:00
24130,36f66ebfbd144aa6fb39338bd87b0bf9,574498be1ecac5af243037bfcc6f8b22,b45d7ebfb7378630f1bcd74b6ff86ff7,2018-02-26 00:11:02,2018-05-29 03:36:33,92 days 03:25:31
29936,440b5d8129165b7e04b3c37e3a5e93d7,6d01fb6c593d9afd887262570b656f9e,9baf5cb77970f539089d09a38bcec5c3,2017-05-21 08:01:31,2017-08-03 09:31:38,74 days 01:30:07
20220,2e5dc86c8c4aa663549caf5e31de840d,02f8dd9b448d3ec2c73e65a22da1121c,7e93a43ef30c4f03f38b393420bc753a,2017-02-03 00:04:49,2017-04-13 10:56:48,69 days 10:51:59


In [117]:
extreme_items = items_orders.assign(
    shipping_window=shipping_window
).nlargest(10, "shipping_window")

extreme_details = extreme_items.merge(
    orders[[
        "order_id",
        "order_status",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]],
    on="order_id",
    how="left"
)

extreme_details[[
    "order_id",
    "seller_id",
    "order_purchase_timestamp",
    "shipping_limit_date",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "shipping_window"
]]

,order_id,seller_id,order_purchase_timestamp,shipping_limit_date,order_status,order_delivered_carrier_date,order_delivered_customer_date,shipping_window
0,13bdf405f961a6deec817d817f5c6624,7a241947449cc45dbfda4f9d0798d9d0,2017-03-16 02:30:51,2020-02-05 03:30:51,canceled,NaT,NaT,1056 days 01:00:00
1,9c94a4ea2f7876660fa6f1b59b69c8e6,7a241947449cc45dbfda4f9d0798d9d0,2017-03-14 19:23:22,2020-02-03 20:23:22,shipped,2017-03-16 14:31:15,NaT,1056 days 01:00:00
2,c2bb89b5c1dd978d507284be78a04cb2,7a241947449cc45dbfda4f9d0798d9d0,2017-05-23 22:28:36,2020-04-09 22:35:08,delivered,2017-05-29 02:03:28,2017-06-09 13:35:54,1052 days 00:06:32
3,c2bb89b5c1dd978d507284be78a04cb2,7a241947449cc45dbfda4f9d0798d9d0,2017-05-23 22:28:36,2020-04-09 22:35:08,delivered,2017-05-29 02:03:28,2017-06-09 13:35:54,1052 days 00:06:32
4,69d126e78947276280838ee9361f5505,88af55b4a7ca402b27df16f7c7c9b5d2,2017-03-30 15:23:23,2017-08-25 15:32:11,processing,NaT,NaT,148 days 00:08:48
5,40dc2ba6f322a17626aac6244332828c,1354d51653f645349064725ed204e85e,2017-10-05 21:39:05,2018-01-10 22:49:21,delivered,2017-10-09 12:13:58,2017-10-13 13:49:07,97 days 01:10:16
6,c72727d29cde4cf870d569bf65edabfd,fce62094ffe6a4009188ec44e681dfdd,2017-02-07 18:01:15,2017-05-12 17:01:15,delivered,2017-02-10 05:32:55,2017-02-14 14:27:45,93 days 23:00:00
7,36f66ebfbd144aa6fb39338bd87b0bf9,b45d7ebfb7378630f1bcd74b6ff86ff7,2018-02-26 00:11:02,2018-05-29 03:36:33,canceled,NaT,NaT,92 days 03:25:31
8,440b5d8129165b7e04b3c37e3a5e93d7,9baf5cb77970f539089d09a38bcec5c3,2017-05-21 08:01:31,2017-08-03 09:31:38,delivered,2017-06-06 12:26:27,2017-06-13 06:51:53,74 days 01:30:07
9,2e5dc86c8c4aa663549caf5e31de840d,7e93a43ef30c4f03f38b393420bc753a,2017-02-03 00:04:49,2017-04-13 10:56:48,processing,NaT,NaT,69 days 10:51:59


In [118]:
long_windows = items_orders.assign(
    shipping_window=shipping_window
)

long_windows = long_windows.loc[
    long_windows["shipping_window"] > pd.Timedelta(days=30)
]

In [119]:
long_windows = long_windows.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

In [120]:
{
    "records_over_30_days": len(long_windows),
    "percentage": round(len(long_windows) / len(items_orders) * 100, 2),
    "unique_orders": long_windows["order_id"].nunique(),
    "status_by_item": long_windows["order_status"].value_counts(),
    "status_by_order": long_windows.drop_duplicates(
        "order_id"
    )["order_status"].value_counts()
}

{'records_over_30_days': 239,
 'percentage': 0.21,
 'unique_orders': 209,
 'status_by_item': order_status
 delivered     215
 processing     11
 canceled        7
 shipped         4
 invoiced        2
 Name: count, dtype: int64,
 'status_by_order': order_status
 delivered     187
 processing     10
 canceled        7
 shipped         3
 invoiced        2
 Name: count, dtype: int64}

In [121]:
long_windows = long_windows.merge(
    orders[["order_id", "order_delivered_carrier_date"]],
    on="order_id",
    how="left"
)

long_windows["handed_over_late"] = (
    long_windows["order_delivered_carrier_date"]
    > long_windows["shipping_limit_date"]
)

{
    "records_with_carrier_date": long_windows["order_delivered_carrier_date"].notna().sum(),
    "records_handed_over_after_deadline": long_windows["handed_over_late"].sum(),
    "records_handed_over_by_deadline": (
        long_windows["order_delivered_carrier_date"].notna()
        & ~long_windows["handed_over_late"]
    ).sum()
}

{'records_with_carrier_date': np.int64(220),
 'records_handed_over_after_deadline': np.int64(20),
 'records_handed_over_by_deadline': np.int64(200)}

In [122]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [123]:
orders[[
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].groupby("order_status").agg(
    total_orders=("order_status", "size"),
    carrier_date_missing=("order_delivered_carrier_date", lambda x: x.isna().sum()),
    customer_date_missing=("order_delivered_customer_date", lambda x: x.isna().sum()),
    estimated_date_missing=("order_estimated_delivery_date", lambda x: x.isna().sum())
)

,total_orders,carrier_date_missing,customer_date_missing,estimated_date_missing
order_status,,,,
approved,2,2,2,0
canceled,625,550,619,0
created,5,5,5,0
delivered,96478,2,8,0
invoiced,314,314,314,0
processing,301,301,301,0
shipped,1107,0,1107,0
unavailable,609,609,609,0


In [124]:
delivered_orders = orders.loc[
    orders["order_status"] == "delivered"
].copy()

delivered_orders["delivery_delay"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
)

delivered_orders["delivery_delay"].describe()

count                        96470
mean     -12 days +19:43:29.937028
std        10 days 04:25:28.179310
min            -147 days +23:36:47
25%      -17 days +18:08:32.750000
50%             -12 days +01:14:44
75%              -7 days +14:38:40
max              188 days 23:24:07
Name: delivery_delay, dtype: object

In [125]:
delay_columns = [
    "order_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_delay"
]

latest_deliveries = delivered_orders.nlargest(
    10, "delivery_delay"
)[delay_columns]

earliest_deliveries = delivered_orders.nsmallest(
    10, "delivery_delay"
)[delay_columns]

latest_deliveries

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay
55619,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,188 days 23:24:07
19590,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,181 days 14:36:39
11399,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,175 days 20:51:31
81401,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,167 days 17:00:07
89130,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,166 days 14:00:04
61610,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,165 days 15:12:50
68769,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,162 days 17:14:25
40847,d24e8541128cea179a11a65176e0a96f,2017-06-12 13:14:11,2017-12-04 18:36:29,2017-06-26,161 days 18:36:29
38509,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,161 days 14:38:21
54480,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,159 days 14:38:18


In [126]:
delivered_orders.loc[
    delivered_orders["delivery_delay"] > pd.Timedelta(days=30),
    "order_delivered_customer_date"
].dt.date.value_counts().head(10)

order_delivered_customer_date
2017-09-19    27
2018-05-07    11
2018-04-19     9
2018-05-14     7
2018-05-03     7
2018-04-03     6
2018-04-21     5
2018-05-08     5
2018-01-23     4
2018-04-23     4
Name: count, dtype: int64

In [127]:
earliest_deliveries

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay
40094,0607f0efea4b566f1eb8f7d3c2397320,2018-03-06 09:47:07,2018-03-09 23:36:47,2018-08-03,-147 days +23:36:47
15791,c72727d29cde4cf870d569bf65edabfd,2017-02-07 18:01:15,2017-02-14 14:27:45,2017-07-04,-140 days +14:27:45
57160,eec7f369423b033e549c02f3c5381205,2018-02-06 20:44:56,2018-02-27 16:35:43,2018-07-12,-135 days +16:35:43
86444,c2bb89b5c1dd978d507284be78a04cb2,2017-05-23 22:28:36,2017-06-09 13:35:54,2017-10-11,-124 days +13:35:54
67488,40dc2ba6f322a17626aac6244332828c,2017-10-05 21:39:05,2017-10-13 13:49:07,2018-01-30,-109 days +13:49:07
79449,1a695d543b7302aa9446c8d5fbd632bf,2017-12-16 10:32:49,2017-12-28 22:18:23,2018-03-22,-84 days +22:18:23
62706,39e0115911bf404857e14baa7f097feb,2018-01-20 07:48:16,2018-02-01 01:52:34,2018-04-25,-83 days +01:52:34
95849,559eea5a72341a4c82dbce9884277cb7,2017-11-17 10:24:03,2017-11-30 16:12:50,2018-02-16,-78 days +16:12:50
63537,38930f76efb00b138f4d632e4d557341,2018-01-28 13:47:42,2018-02-08 19:21:39,2018-04-27,-78 days +19:21:39
37491,c5132855100a12d63ed4e8ae05f9594d,2017-10-13 14:45:03,2017-10-25 20:22:10,2018-01-11,-78 days +20:22:10


Delivery Performance

- Analyzed 96,470 delivered orders with valid timestamps.
- Orders were typically delivered around 12 days earlier than estimated.
- Delivery delays ranged from 147 days early to 189 days late.
- Some extreme delays shared the same delivery dates, particularly September 19, 2017.
- Missing timestamps were mostly consistent with order status.

**Decision:** Preserve the original dates and extreme values. Exclude invalid timestamp sequences only when they affect a specific analysis.

In [128]:
{
    "rows": len(customers),
    "missing_values": customers.isna().sum().to_dict(),
    "duplicate_customer_ids": customers["customer_id"].duplicated().sum(),
    "unique_customer_ids": customers["customer_id"].nunique(),
    "unique_customers": customers["customer_unique_id"].nunique(),
    "repeat_customer_records": customers["customer_unique_id"].duplicated().sum()
}

{'rows': 99441,
 'missing_values': {'customer_id': 0,
  'customer_unique_id': 0,
  'customer_zip_code_prefix': 0,
  'customer_city': 0,
  'customer_state': 0},
 'duplicate_customer_ids': np.int64(0),
 'unique_customer_ids': 99441,
 'unique_customers': 96096,
 'repeat_customer_records': np.int64(3345)}

 Customer Audit

- 99,441 records with no missing values.
- `customer_id` is unique across all records.
- 96,096 unique customers based on `customer_unique_id`.
- 3,345 records belong to customers with multiple records.

**Decision:** Keep the customer records as they are. Use `customer_unique_id` when analyzing actual customer behavior and retention.

In [129]:
{
    "rows": len(sellers),
    "missing_values": sellers.isna().sum().to_dict(),
    "duplicate_seller_ids": sellers["seller_id"].duplicated().sum(),
    "unique_seller_ids": sellers["seller_id"].nunique(),
    "unique_zip_prefixes": sellers["seller_zip_code_prefix"].nunique(),
    "unique_cities": sellers["seller_city"].nunique(),
    "unique_states": sellers["seller_state"].nunique()
}

{'rows': 3095,
 'missing_values': {'seller_id': 0,
  'seller_zip_code_prefix': 0,
  'seller_city': 0,
  'seller_state': 0},
 'duplicate_seller_ids': np.int64(0),
 'unique_seller_ids': 3095,
 'unique_zip_prefixes': 2246,
 'unique_cities': 611,
 'unique_states': 23}

 Seller Audit

- 3,095 sellers with no missing values.
- `seller_id` is unique.
- Sellers are distributed across 23 states and 611 cities.
- Multiple sellers can share the same ZIP prefix.

**Decision:** No cleaning required.

In [130]:
category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [131]:
{
    "rows": len(category_translation),
    "missing_values": category_translation.isna().sum().to_dict(),
    "duplicate_categories": category_translation["product_category_name"].duplicated().sum(),
    "duplicate_translations": category_translation["product_category_name_english"].duplicated().sum(),
    "unique_source_categories": category_translation["product_category_name"].nunique(),
    "unique_english_categories": category_translation["product_category_name_english"].nunique()
}

{'rows': 71,
 'missing_values': {'product_category_name': 0,
  'product_category_name_english': 0},
 'duplicate_categories': np.int64(0),
 'duplicate_translations': np.int64(0),
 'unique_source_categories': 71,
 'unique_english_categories': 71}

 Category Translation Audit

- 71 categories with no missing values.
- Source categories and English translations are both unique.
- The translation mapping is one-to-one.

**Decision:** No cleaning required.

In [132]:
orders[
    [
        "invalid_timestamp_sequence",
        "carrier_before_approval",
        "delivery_before_carrier"
    ]
].sum()

invalid_timestamp_sequence    1382
carrier_before_approval       1359
delivery_before_carrier         23
dtype: int64

### Data Cleaning ###

Based on the audit findings, the raw data will be cleaned while preserving valid but unusual records. Records will only be excluded from specific analyses when data-quality issues affect the calculation.

In [133]:
orders_clean = orders.copy()

In [134]:
orders_clean.shape

(99441, 11)

Orders Cleaning

- Preserved all 99,441 orders.
- Kept the original timestamp values.
- Retained the timestamp-quality flags for analysis.
- Orders with invalid timestamp sequences will only be excluded from analyses that depend on timestamp order.

In [135]:
order_items_clean = order_items.copy()

In [136]:
order_items_clean.shape

(112650, 7)

In [137]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"]
)

In [138]:
order_items_clean["shipping_limit_date"].dtype

dtype('<M8[us]')

 Order Items Cleaning

- Preserved all 112,650 order-item records.
- Converted `shipping_limit_date` to datetime.
- Kept zero freight values and extreme shipping windows.
- No missing or invalid price/freight values were found.

**Decision:** No records were removed.

In [139]:
payments_clean = payments.copy()

In [140]:
payments_clean.shape

(103886, 5)

 Payments Cleaning

- Preserved all 103,886 payment records.
- No missing or invalid payment values were found.
- Kept zero-value and unusual payment sequences.

**Decision:** No records were removed.

In [141]:
reviews_clean = reviews.copy()

In [142]:
reviews_clean.shape

(99224, 7)

Reviews Cleaning

- Preserved all 99,224 review records.
- Kept repeated `review_id` values and multiple reviews per order.
- Missing review titles and messages were preserved.

**Decision:** No records were removed.

In [143]:
products_clean = products.copy()

In [144]:
products_clean.shape

(32951, 9)

In [145]:
products_clean = products_clean.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [146]:
products_clean.shape

(32951, 10)

In [147]:
products_clean["product_category_name_english"].isna().sum()

np.int64(623)

 Products Cleaning

- Preserved all 32,951 products.
- Added English category names using the translation table.
- 623 products have no English category because their original category is missing or untranslated.
- Missing product attributes were preserved.

**Decision:** No products were removed or artificially imputed.

In [148]:
geolocation_clean = geolocation.drop_duplicates().copy()

In [149]:
geolocation_clean.shape

(738332, 5)

Geolocation Cleaning

- Removed exact duplicate rows.
- Reduced the table from 1,000,163 to 738,332 rows.
- Kept multiple locations for the same ZIP prefix.

**Decision:** Deduplicate only exact rows. Do not deduplicate by ZIP prefix.

In [150]:
customers_clean = customers.copy()

In [151]:
customers_clean.shape

(99441, 5)

 Customers Cleaning

- Preserved all 99,441 customer records.
- No missing values or duplicate `customer_id` values.
- Kept repeated `customer_unique_id` values for retention analysis.

**Decision:** No records were removed.

In [152]:
sellers_clean = sellers.copy()

In [153]:
sellers_clean.shape

(3095, 4)

 Sellers Cleaning

- Preserved all 3,095 seller records.
- No missing values or duplicate `seller_id` values.
- Multiple sellers can share the same ZIP prefix.

**Decision:** No records were removed.

In [154]:
category_translation_clean = category_translation.copy()
category_translation_clean.shape

(71, 2)

 Category Translation Cleaning

- Preserved all 71 category translations.
- No missing values or duplicate mappings.

**Decision:** No records were removed.

In [155]:
cleaned_tables = {
    "orders": orders_clean,
    "order_items": order_items_clean,
    "payments": payments_clean,
    "reviews": reviews_clean,
    "products": products_clean,
    "geolocation": geolocation_clean,
    "customers": customers_clean,
    "sellers": sellers_clean,
    "category_translation": category_translation_clean
}

for name, df in cleaned_tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 11)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 10)
geolocation: (738332, 5)
customers: (99441, 5)
sellers: (3095, 4)
category_translation: (71, 2)


In [156]:
for name, df in cleaned_tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")
    print(missing if not missing.empty else "No missing values")


orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

order_items
No missing values

payments
No missing values

reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

products
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    623
dtype: int64

geolocation
No missing values

customers
No missing values

sellers
No missing values

category_translation
No missing values


Missing Values Validation

- Remaining missing values match the issues identified during the audit.
- Order timestamp gaps are consistent with different order statuses.
- Missing review text and product attributes were preserved.
- No unexpected missing values were introduced during cleaning.

**Decision:** No further missing-value treatment was required.

In [157]:
checks = {
    "orders → customers": orders_clean["customer_id"].isin(
        customers_clean["customer_id"]
    ).all(),

    "order_items → orders": order_items_clean["order_id"].isin(
        orders_clean["order_id"]
    ).all(),

    "order_items → products": order_items_clean["product_id"].isin(
        products_clean["product_id"]
    ).all(),

    "order_items → sellers": order_items_clean["seller_id"].isin(
        sellers_clean["seller_id"]
    ).all(),

    "payments → orders": payments_clean["order_id"].isin(
        orders_clean["order_id"]
    ).all(),

    "reviews → orders": reviews_clean["order_id"].isin(
        orders_clean["order_id"]
    ).all(),
}

checks

{'orders → customers': np.True_,
 'order_items → orders': np.True_,
 'order_items → products': np.True_,
 'order_items → sellers': np.True_,
 'payments → orders': np.True_,
 'reviews → orders': np.True_}

Relationship Validation

- All key relationships between the cleaned tables remain valid.
- No orphaned orders, products, sellers, payments, or reviews were introduced.

**Result:** Referential integrity passed for all validated relationships.

In [158]:
key_checks = {
    "orders.order_id unique": orders_clean["order_id"].is_unique,
    "customers.customer_id unique": customers_clean["customer_id"].is_unique,
    "sellers.seller_id unique": sellers_clean["seller_id"].is_unique,
    "products.product_id unique": products_clean["product_id"].is_unique,
    "order_items composite key unique": ~order_items_clean.duplicated(
        ["order_id", "order_item_id"]
    ).any(),
    "payments composite key unique": ~payments_clean.duplicated(
        ["order_id", "payment_sequential"]
    ).any(),
}

key_checks

{'orders.order_id unique': True,
 'customers.customer_id unique': True,
 'sellers.seller_id unique': True,
 'products.product_id unique': True,
 'order_items composite key unique': np.True_,
 'payments composite key unique': np.True_}

Final Validation Summary

- All cleaned tables passed the structural validation checks.
- Key columns and composite keys remain unique.
- Referential integrity is preserved across the dataset.
- Remaining missing values and unusual records were previously reviewed and intentionally retained.

**Result:** The cleaned dataset is ready to be saved as processed data.

In [159]:

# Review Text Analysis: Initial Validation
# Reuses reviews_clean from the existing dataset audit notebook.

import pandas as pd
import unicodedata

text_columns = ["review_comment_title", "review_comment_message"]

review_text = reviews_clean[
    ["review_id", "order_id", "review_score"] + text_columns
].copy()

# Normalize missing text without modifying the original cleaned DataFrame.
for col in text_columns:
    review_text[col] = review_text[col].fillna("").astype(str).str.strip()

review_text["combined_text"] = (
    review_text["review_comment_title"] + " " +
    review_text["review_comment_message"]
).str.strip()

review_text["has_text"] = review_text["combined_text"].ne("")

# Check for common Unicode replacement/corruption indicators.
def inspect_text(value):
    if not value:
        return "empty"
    if "\ufffd" in value:
        return "replacement_character_found"
    return "text_present"

review_text["text_check"] = review_text["combined_text"].apply(inspect_text)

print("REVIEW TEXT VALIDATION")
print("=" * 50)
print(f"Review records: {len(review_text):,}")
print(f"Records with usable text: {review_text['has_text'].sum():,}")
print(f"Records without text: {(~review_text['has_text']).sum():,}")
print(
    "Records containing Unicode replacement characters:",
    review_text["text_check"].eq("replacement_character_found").sum()
)

print("\nREPEATED REVIEW ID CHECK")
print("=" * 50)
print(
    "Rows with a repeated review_id:",
    review_text["review_id"].duplicated(keep=False).sum()
)
print(
    "Distinct repeated review IDs:",
    review_text.loc[
        review_text["review_id"].duplicated(keep=False),
        "review_id"
    ].nunique()
)

print("\nSAMPLE PORTUGUESE TEXT")
print("=" * 50)

sample_reviews = review_text.loc[
    review_text["has_text"],
    ["review_score", "review_comment_title",
     "review_comment_message", "combined_text"]
].head(10)

for _, row in sample_reviews.iterrows():
    print(f"\nScore: {row['review_score']}")
    print(f"Title: {row['review_comment_title']}")
    print(f"Message: {row['review_comment_message']}")

print("\nUNICODE CHARACTER INSPECTION")
print("=" * 50)

sample_text = next(
    (x for x in review_text["combined_text"] if x),
    ""
)

print("Python representation:", repr(sample_text))
print(
    "Unicode name examples:",
    [
        (char, unicodedata.name(char, "UNKNOWN"))
        for char in sample_text
        if ord(char) > 127
    ][:15]
)


REVIEW TEXT VALIDATION
Review records: 99,224
Records with usable text: 42,687
Records without text: 56,537
Records containing Unicode replacement characters: 0

REPEATED REVIEW ID CHECK
Rows with a repeated review_id: 1603
Distinct repeated review IDs: 789

SAMPLE PORTUGUESE TEXT

Score: 5
Title: 
Message: Recebi bem antes do prazo estipulado.

Score: 5
Title: 
Message: Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa

Score: 4
Title: recomendo
Message: aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho

Score: 4
Title: 
Message: Mas um pouco ,travando...pelo valor ta Boa.

Score: 5
Title: Super recomendo
Message: Vendedor confiável, produto ok e entrega antes do prazo.

Score: 2
Title: 
Message: GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E ESSA COMPRA AGORA ME DECPCIONOU

Score: 1
Title: Não chegou meu produt

In [161]:

# Step 2: Rule-Based Review Theme Classification
# Reuses review_text from the previous validation cell.
# Reviews can belong to multiple themes.

import re
import unicodedata
import pandas as pd

def normalize_portuguese(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )
    text = re.sub(r"\s+", " ", text)
    return text

# Create a working copy. Original Portuguese text remains unchanged.
reviews_analysis = review_text.copy()

reviews_analysis["normalized_text"] = (
    reviews_analysis["combined_text"].apply(normalize_portuguese)
)

# Initial keyword rules. These will be refined after inspecting examples.
theme_keywords = {
    "non_delivery": [
        "nao chegou", "nao recebi", "nunca recebi",
        "nao foi entregue", "produto nao entregue",
        "pedido nao chegou", "ainda nao recebi",
        "nao entregaram"
    ],
    "late_delivery": [
        "atraso", "atrasado", "atrasada", "atrasou",
        "demorou para entregar", "demora na entrega",
        "demorou a chegar", "depois do prazo",
        "fora do prazo", "prazo vencido",
        "entrega tardia", "entregue com atraso"
    ],
    "missing_items": [
        "faltou", "faltando", "item faltando",
        "produto faltando", "veio incompleto",
        "pedido incompleto", "quantidade errada",
        "faltaram", "peca faltando"
    ],
    "wrong_product": [
        "produto errado", "pedido errado",
        "cor errada", "cor diferente",
        "tamanho errado", "modelo errado",
        "veio outro produto", "produto diferente",
        "mandaram errado", "envio errado"
    ],
    "product_defect_or_damage": [
        "com defeito", "veio quebrado", "produto quebrado",
        "produto danificado", "nao funciona",
        "parou de funcionar", "mal funcionamento",
        "baixa qualidade", "veio estragado",
        "peca quebrada", "produto com problema"
    ],
    "packaging": [
        "embalagem", "caixa amassada", "mal embalado",
        "pacote aberto", "caixa aberta",
        "produto mal acondicionado"
    ],
    "support_or_refund": [
        "atendimento", "suporte", "nao respondem",
        "sem resposta", "nao responderam",
        "nao consigo contato", "reembolso",
        "devolucao", "troca do produto",
        "vendedor nao responde", "sac"
    ],
    "authenticity_concerns": [
        "falsificado", "falso", "replica",
        "nao e original", "nao parece original",
        "produto pirata", "duvida da autenticidade"
    ],
    "product_description_mismatch": [
        "diferente da foto", "diferente das fotos",
        "nao corresponde a descricao",
        "nao condiz com a descricao",
        "diferente do anunciado", "foto enganosa",
        "descricao incorreta", "nao e como anunciado",
        "nao veio conforme anunciado"
    ],
    "freight_or_shipping_cost": [
        "frete caro", "frete alto", "frete abusivo",
        "valor do frete", "preco do frete",
        "frete muito caro", "custo do frete"
    ]
}

# Classify only reviews with usable text.
for theme, keywords in theme_keywords.items():
    reviews_analysis[theme] = (
        reviews_analysis["normalized_text"].apply(
            lambda text: any(keyword in text for keyword in keywords)
        )
        & reviews_analysis["has_text"]
    )

# Summary: themes can overlap, so percentages will not add up to 100%.
text_reviews = reviews_analysis.loc[
    reviews_analysis["has_text"]
].copy()

summary_rows = []

for theme in theme_keywords:
    matched = text_reviews.loc[text_reviews[theme]]
    count = len(matched)

    summary_rows.append({
        "theme": theme,
        "review_count": count,
        "pct_of_text_reviews": round(
            count / len(text_reviews) * 100, 2
        ) if len(text_reviews) else 0,
        "average_score": round(
            matched["review_score"].mean(), 2
        ) if count else None,
        "pct_low_rating_1_or_2": round(
            matched["review_score"].le(2).mean() * 100, 2
        ) if count else None
    })

theme_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("review_count", ascending=False)
    .reset_index(drop=True)
)

print("REVIEW THEME SUMMARY")
print("=" * 85)
display(theme_summary)

print("\nTEXT REVIEW COVERAGE")
print("=" * 85)
print(f"All review records: {len(reviews_analysis):,}")
print(f"Reviews with text: {len(text_reviews):,}")
print(
    "Reviews matching at least one theme:",
    int(text_reviews[list(theme_keywords)].any(axis=1).sum())
)

# Inspect examples for manual validation.
# These are candidate matches, not confirmed labels.
for theme in theme_keywords:
    examples = (
        text_reviews.loc[
            text_reviews[theme],
            ["review_score", "combined_text"]
        ]
        .sort_values("review_score")
        .head(3)
    )

    print(f"\n{'=' * 70}\nTHEME: {theme}\n{'=' * 70}")

    if examples.empty:
        print("No matches found.")
    else:
        for _, row in examples.iterrows():
            print(f"Score: {row['review_score']}")
            print(f"Review: {row['combined_text']}\n")


REVIEW THEME SUMMARY


,theme,review_count,pct_of_text_reviews,average_score,pct_low_rating_1_or_2
0,non_delivery,3528,8.26,1.45,86.28
1,support_or_refund,1513,3.54,3.10,42.96
2,late_delivery,628,1.47,2.21,62.90
3,product_defect_or_damage,622,1.46,1.79,76.53
4,missing_items,616,1.44,2.10,63.15
5,packaging,479,1.12,2.97,39.67
6,wrong_product,363,0.85,1.72,76.31
7,authenticity_concerns,196,0.46,1.49,84.18
8,freight_or_shipping_cost,100,0.23,3.06,29.00
9,product_description_mismatch,71,0.17,1.92,67.61



TEXT REVIEW COVERAGE
All review records: 99,224
Reviews with text: 42,687
Reviews matching at least one theme: 7442

THEME: non_delivery
Score: 1
Review: Não chegou meu produto Péssimo

Score: 1
Review: Ainda não recebi a base Dermacol 223

Score: 1
Review: Por enquanto está péssimo Não recebi o produto, vcs dizem que foi entregue dia 21 mas até agora nada, e não consigo contato com ninguém.


THEME: late_delivery
Score: 1
Review: Entrega atrasada.

Score: 1
Review: Comprei o produto mais não foi entregue, toda vez que entro em contato me garantem um retorno de 48 horas de 1 dia mais nunca retornam a entrega ja esta 14 dias em atraso.

Score: 1
Review: PEDIDO NÃO ENTREGUE NENHUMA ESCLARECIMENTO SOBRE ATRASO DA ENTREGA.


THEME: missing_items
Score: 1
Review: recebi somente 1 controle Midea Split ESTILO.
Faltou Controle Remoto para Ar Condicionado Consul

Score: 1
Review: Estou com sério problema fiz uma super compra pra vim faltando o meu produto ... e além do mais pagar um frete absu

In [162]:

# Step 4: Manually validate theme classification
# Reuses reviews_analysis and theme_keywords.

for theme in theme_keywords:
    matched = (
        reviews_analysis.loc[
            reviews_analysis[theme],
            ["review_score", "combined_text"]
        ]
        .drop_duplicates(subset=["combined_text"])
    )

    if matched.empty:
        print(f"\n{theme}: No matches")
        continue

    # Include low- and high-rated examples to expose false positives.
    low = matched.loc[matched["review_score"] <= 2].head(3)
    high = matched.loc[matched["review_score"] >= 4].head(3)

    examples = pd.concat([low, high]).drop_duplicates(
        subset=["combined_text"]
    ).head(5)

    print(f"\n{'=' * 75}")
    print(f"THEME: {theme} | Total matches: {len(matched):,}")
    print("=" * 75)

    for _, row in examples.iterrows():
        print(f"Score: {row['review_score']}")
        print(f"Text: {row['combined_text']}\n")



THEME: non_delivery | Total matches: 3,250
Score: 1
Text: Não chegou meu produto Péssimo

Score: 1
Text: Sempre compro pela Internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. No stark o prazo máximo já se esgotou e ainda não recebi o produto.

Score: 1
Text: O produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, baseado no prazo. Moro na Bahia e ele em Cuiabá sozinho. Agora, a casa está vazia.

Score: 5
Text: Muito satisfeito entrega super rápida.. Show ..parabéns.. eu vacilei e comprei um celular nas casas Bahia tem quase um mês e não entregaram..

Score: 4
Text: Conforme já solicitei, preciso fazer a devolução. Ele não foi entregue no prazo que eu precisa. Já solicitei por aqui e já entrei em contato com o vendedor. Estou no aguardo para fazer a devolução.


THEME: late_delivery | Total matches: 604
Score: 1
Text: Demora na entrega, detestei o atendimento e NUNCA mais compro. baratheon quando ve

In [163]:

# Step 6: Refined rule-based classification
# Reuses reviews_analysis and review_text from previous cells.

refined_keywords = {
    "non_delivery": [
        "nao chegou meu produto", "nao chegou o produto",
        "nao recebi", "ainda nao recebi",
        "nunca recebi", "pedido nao chegou",
        "produto nao foi entregue", "nao foi entregue",
        "nao entregaram", "aguardando a entrega",
        "produto nao chegou"
    ],

    "late_delivery": [
        "atraso na entrega", "entrega atrasada",
        "entrega com atraso", "atrasou a entrega",
        "demora na entrega", "demora para entregar",
        "demorou para chegar", "depois do prazo",
        "fora do prazo", "prazo de entrega",
        "atraso na data de entrega"
    ],

    "missing_order_items": [
        "faltou produto", "faltaram produtos",
        "faltou um produto", "faltou uma unidade",
        "recebi somente", "recebi apenas",
        "nao veio junto", "nao enviaram todos",
        "pedido incompleto", "veio incompleto",
        "nao recebi todos", "faltou uma peca"
    ],

    "wrong_product": [
        "produto errado", "produto diferente do que pedi",
        "veio outro produto", "me enviaram errado",
        "foi enviado o produto errado", "modelo veio diferente",
        "cor diferente da que pedi", "nao e o que pedi",
        "nao foi o que comprei", "mandaram errado"
    ],

    "product_defect_or_damage": [
        "com defeito", "veio quebrado", "produto quebrado",
        "produto danificado", "nao funciona",
        "parou de funcionar", "mal funcionamento",
        "baixa qualidade", "veio estragado",
        "veio avariado", "veio com defeito",
        "nao esta funcionando"
    ],

    "packaging_damage": [
        "caixa amassada", "embalagem amassada",
        "embalagem danificada", "pacote aberto",
        "caixa aberta", "embalagem rasgada",
        "produto mal embalado", "embalagem muito fraca",
        "produto chegou avariado por causa da embalagem"
    ],

    "support_complaint": [
        "sem resposta", "nao responderam",
        "nao responde", "nao consigo contato",
        "nao consigo falar", "pessimo atendimento",
        "atendimento ruim", "nao me deram retorno",
        "aguardo retorno", "nao deram satisfacao",
        "nao solucionaram", "nao resolveram meu problema"
    ],

    "refund_or_return_problem": [
        "reembolso nao recebido", "nao recebi o reembolso",
        "reembolso pendente", "estorno nao recebido",
        "nao devolveram meu dinheiro",
        "aguardando meu dinheiro", "nao fizeram o estorno",
        "solicitei a devolucao e nao",
        "pedi reembolso e sem resposta"
    ],

    "authenticity_concerns": [
        "produto pirata", "produto falsificado",
        "nao e original", "nao parece original",
        "replica do original", "produto falso",
        "nao se trata de original"
    ],

    "product_description_mismatch": [
        "diferente da foto", "diferente das fotos",
        "diferente do anunciado", "nao corresponde a descricao",
        "nao condiz com a descricao",
        "nao e como anunciado", "foto do anuncio",
        "diferente do produto anunciado"
    ],

    "freight_or_shipping_cost": [
        "frete caro", "frete muito caro",
        "frete alto", "frete abusivo",
        "custo do frete", "valor do frete",
        "preco do frete", "pagar dois fretes",
        "pagar 2 fretes"
    ]
}

# Reset old labels so the refined summary uses only the new definitions.
old_theme_columns = list(theme_keywords.keys())
reviews_analysis = reviews_analysis.drop(
    columns=[c for c in old_theme_columns if c in reviews_analysis.columns],
    errors="ignore"
)

theme_keywords = refined_keywords

for theme, keywords in theme_keywords.items():
    reviews_analysis[theme] = (
        reviews_analysis["normalized_text"].apply(
            lambda text: any(keyword in text for keyword in keywords)
        )
        & reviews_analysis["has_text"]
    )

text_reviews = reviews_analysis.loc[
    reviews_analysis["has_text"]
].copy()

summary_rows = []

for theme in theme_keywords:
    matched = text_reviews.loc[text_reviews[theme]]
    count = len(matched)

    summary_rows.append({
        "theme": theme,
        "review_count": count,
        "pct_of_text_reviews": round(
            count / len(text_reviews) * 100, 2
        ),
        "average_score": round(
            matched["review_score"].mean(), 2
        ) if count else None,
        "pct_low_rating_1_or_2": round(
            matched["review_score"].le(2).mean() * 100, 2
        ) if count else None
    })

theme_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("review_count", ascending=False)
    .reset_index(drop=True)
)

display(theme_summary)

print("\nSample matches for manual validation")

for theme in theme_keywords:
    examples = reviews_analysis.loc[
        reviews_analysis[theme],
        ["review_score", "combined_text"]
    ].drop_duplicates(subset=["combined_text"]).head(3)

    print(f"\n{theme}:")
    for _, row in examples.iterrows():
        print(f"Score {row['review_score']}: {row['combined_text']}")


,theme,review_count,pct_of_text_reviews,average_score,pct_low_rating_1_or_2
0,non_delivery,3054,7.15,1.43,86.90
1,late_delivery,833,1.95,2.99,43.22
2,product_defect_or_damage,639,1.50,1.79,76.53
3,missing_order_items,426,1.00,1.69,77.23
4,support_complaint,214,0.50,1.27,91.59
5,wrong_product,192,0.45,1.49,85.42
6,freight_or_shipping_cost,104,0.24,3.09,28.85
7,authenticity_concerns,97,0.23,1.48,83.51
8,product_description_mismatch,82,0.19,1.89,69.51
9,packaging_damage,21,0.05,2.52,47.62



Sample matches for manual validation

non_delivery:
Score 1: Não chegou meu produto Péssimo
Score 1: Sempre compro pela Internet e a entrega ocorre antes do prazo combinado, que acredito ser o prazo máximo. No stark o prazo máximo já se esgotou e ainda não recebi o produto.
Score 1: O produto não chegou no prazo estipulado e causou transtorno, pq programei a viagem de férias do meu filho, baseado no prazo. Moro na Bahia e ele em Cuiabá sozinho. Agora, a casa está vazia.

late_delivery:
Score 3: Entrega atrasou A entrega foi dividida em duas. Não houve comunicado do loja. Cheguei a pensar que só haviam enviado parte do produto.
Score 1: Demora na entrega, detestei o atendimento e NUNCA mais compro. baratheon quando vendia por eles mesmos era bem melhor!
Score 3: O preço foi ótimo, o prazo de entrega que é longo (mais de 1 mês para entregar). Comprei 2 cartuchos e entregaram apenas 1, reclamei e enviaram o outro ainda dentro do prazo de entrega.

missing_order_items:
Score 1: recebi som

In [164]:

# Diagnostic: find refund/return mentions missed by the current rule.
# Reuses reviews_analysis and normalized_text.

refund_terms = [
    "reembolso", "reembolsar", "estorno",
    "devolver meu dinheiro", "dinheiro de volta",
    "devolucao", "devolver o produto",
    "quero meu dinheiro", "solicitei a troca",
    "pedi a troca", "trocar o produto"
]

refund_pattern = "|".join(
    re.escape(normalize_portuguese(term))
    for term in refund_terms
)

refund_candidates = reviews_analysis.loc[
    reviews_analysis["has_text"]
    & reviews_analysis["normalized_text"].str.contains(
        refund_pattern, regex=True, na=False
    ),
    ["review_score", "combined_text", "refund_or_return_problem"]
].copy()

print("Reviews mentioning refund/return-related language:",
      len(refund_candidates))
print("Already classified:",
      int(refund_candidates["refund_or_return_problem"].sum()))
print("Not classified:",
      int((~refund_candidates["refund_or_return_problem"]).sum()))

display(
    refund_candidates.loc[
        ~refund_candidates["refund_or_return_problem"]
    ].head(20)
)


Reviews mentioning refund/return-related language: 807
Already classified: 1
Not classified: 806


,review_score,combined_text,refund_or_return_problem
426,1,Produto pirata! Quando instalei o cartucho de ...,False
443,1,Não foram entregue as duas capas de couro comp...,False
643,1,"Gostaria de receber os produtos corretos, tent...",False
646,1,"meu produto veio com defeito,devolvi conforme ...",False
711,1,comprei tapete de borracha para tr4 mitsubishi...,False
726,3,PRODUTO COM DEFEITO SOLICITEI DEVOLUÇÃO. PRODU...,False
760,1,"MEU PRODUTO AINDA NÃO FOI ENTREGUE, JÁ FIZ MIN...",False
888,1,Estou aguardando posição da empresa para fazer...,False
1020,5,Péssima Recebi o produto que eh falso quero me...,False
1021,1,PRODUTO NÃO VEIO CONFORME ESTAVA NO SITE .\r\n...,False


In [165]:

# Refund and return diagnostic: mention vs potential problem
# Reuses reviews_analysis and normalized_text.

return_terms = [
    "reembolso", "reembolsar", "estorno",
    "devolucao", "devolver o produto",
    "devolver meu dinheiro", "dinheiro de volta",
    "solicitei a troca", "pedi a troca",
    "trocar o produto", "troca do produto"
]

problem_terms = [
    "sem resposta", "nao responderam",
    "nao devolveram", "nao recebi o reembolso",
    "reembolso pendente", "estorno nao recebido",
    "nao fizeram o estorno", "aguardando reembolso",
    "aguardando devolucao", "aguardando a devolucao",
    "aguardando retorno", "ate agora nada",
    "nao consigo contato", "nao deram retorno",
    "pedi reembolso e sem resposta",
    "preciso fazer a devolucao",
    "solicitei devolucao", "solicitei a devolucao",
    "solicitei a troca", "pedi a devolucao"
]

def contains_any(text, terms):
    return any(term in text for term in terms)

reviews_analysis["refund_return_mentioned"] = (
    reviews_analysis["normalized_text"].apply(
        lambda text: contains_any(text, return_terms)
    )
    & reviews_analysis["has_text"]
)

reviews_analysis["refund_return_problem_candidate"] = (
    reviews_analysis["normalized_text"].apply(
        lambda text: contains_any(text, problem_terms)
    )
    & reviews_analysis["refund_return_mentioned"]
)

print("Reviews mentioning return/refund language:",
      int(reviews_analysis["refund_return_mentioned"].sum()))

print("Potential unresolved return/refund cases:",
      int(reviews_analysis["refund_return_problem_candidate"].sum()))

print("\nPotential problem candidates by rating:")
display(
    reviews_analysis.loc[
        reviews_analysis["refund_return_problem_candidate"]
    ].groupby("review_score").size().rename("review_count").to_frame()
)

# Inspect examples with full text before accepting the label.
pd.set_option("display.max_colwidth", None)

candidates = reviews_analysis.loc[
    reviews_analysis["refund_return_problem_candidate"],
    ["review_score", "combined_text"]
].drop_duplicates(subset=["combined_text"])

display(candidates.head(25))


Reviews mentioning return/refund language: 831
Potential unresolved return/refund cases: 135

Potential problem candidates by rating:


,review_count
review_score,
1,89
2,14
3,19
4,10
5,3


,review_score,combined_text
89,1,Pedi reembolso e sem resposta até momento
426,1,"Produto pirata! Quando instalei o cartucho de tinta, a minha impressora indicou que o mesmo estava vazio, e apresentou a mensagem de erro ""!"" em cima do marcador de nível da tinta. Já solicitei a troca."
726,3,PRODUTO COM DEFEITO SOLICITEI DEVOLUÇÃO. PRODUTO COM DEFEITO
1232,4,"Conforme já solicitei, preciso fazer a devolução. Ele não foi entregue no prazo que eu precisa. Já solicitei por aqui e já entrei em contato com o vendedor. Estou no aguardo para fazer a devolução."
1581,1,"Já solicitei devolução do produto. Veio amassado, parece que houve uma queda"
2092,1,"Péssimo atendimento Recebi produto errado à uma semana. Aceitei mas o valor do produto entregue é menor. Pedi reembolso mas até agora nada, faz 9 dias já"
2712,1,PÉSSIMO SERVIÇO Já solicitei a troca porém até o momento nada. Enviaram o produto errado e agora informam que a troca não pode ser feita.
2742,4,solicitei devolução não obtive retorno até o momento
3500,1,Já solicitei a devolução Produto enviado é falaificado
3877,1,Solicitei a devolução Recebi um produto diferente do que foi requisitado. A imagem do anúncio não condiz com o produto recebido.


In [166]:

# Step 7: Refine return/refund labels
# Reuses reviews_analysis and normalized_text.

return_mention_terms = [
    "reembolso", "reembolsar", "estorno",
    "devolucao", "devolucao do dinheiro",
    "devolver meu dinheiro", "dinheiro de volta",
    "devolucao do valor", "devolucao do produto",
    "solicitei a troca", "solicitei troca",
    "pedi a troca", "pedi troca",
    "solicitei a devolucao", "solicitei devolucao",
    "preciso fazer a devolucao", "quero devolver",
    "quero meu dinheiro", "troca do produto"
]

unresolved_terms = [
    "sem resposta", "nao responderam",
    "nao devolveram", "nao recebi o reembolso",
    "reembolso pendente", "estorno nao recebido",
    "nao fizeram o estorno", "aguardando reembolso",
    "aguardando retorno", "ate agora nada",
    "nao consigo contato", "nao deram retorno",
    "nao obtive retorno", "nao obtive resposta",
    "nao solucionaram", "nao resolveram",
    "ate o momento nada", "estou esperando",
    "aguardando a postagem", "troca nao pode ser feita",
    "nao consigo fazer a troca"
]

reviews_analysis["refund_return_mentioned"] = (
    reviews_analysis["normalized_text"].apply(
        lambda text: contains_any(text, return_mention_terms)
    )
    & reviews_analysis["has_text"]
)

reviews_analysis["refund_return_problem_candidate"] = (
    reviews_analysis["normalized_text"].apply(
        lambda text: contains_any(text, unresolved_terms)
    )
    & reviews_analysis["refund_return_mentioned"]
)

# Summarize without claiming every candidate is a confirmed problem.
return_summary = (
    reviews_analysis.loc[reviews_analysis["refund_return_mentioned"]]
    .groupby("review_score")
    .agg(
        reviews_mentioning_return=("review_score", "size"),
        potential_problem_candidates=(
            "refund_return_problem_candidate", "sum"
        )
    )
    .reset_index()
)

print("Reviews mentioning return/refund language:",
      int(reviews_analysis["refund_return_mentioned"].sum()))

print("Potential problem candidates:",
      int(reviews_analysis["refund_return_problem_candidate"].sum()))

display(return_summary)

print("\nPotential problem examples for manual validation:")

display(
    reviews_analysis.loc[
        reviews_analysis["refund_return_problem_candidate"],
        ["review_score", "combined_text"]
    ]
    .drop_duplicates(subset=["combined_text"])
    .head(30)
)


Reviews mentioning return/refund language: 823
Potential problem candidates: 50


,review_score,reviews_mentioning_return,potential_problem_candidates
0,1,631,42
1,2,69,4
2,3,67,1
3,4,27,2
4,5,29,1



Potential problem examples for manual validation:


,review_score,combined_text
89,1,Pedi reembolso e sem resposta até momento
810,1,"Recebi outra boneca, quero devolver e não consigo contato! Jamais compraria novamente dessa loja!"
2092,1,"Péssimo atendimento Recebi produto errado à uma semana. Aceitei mas o valor do produto entregue é menor. Pedi reembolso mas até agora nada, faz 9 dias já"
2712,1,PÉSSIMO SERVIÇO Já solicitei a troca porém até o momento nada. Enviaram o produto errado e agora informam que a troca não pode ser feita.
2742,4,solicitei devolução não obtive retorno até o momento
4955,1,"Comprei o produto é solicitei a troca a targaryen e até o momento não obtive retorno, o produto veio embalado mas com sinal de uso, já solicitei a troca e devolucao do dinheiro e não retornaram"
5547,1,Vou mandar o produto de volta. Estou esperando o andamento da solicitação de devolução que fiz à stark.
5782,1,"Não recebi o produto, já enviei vários e-mail sem resposta..quero meu dinheiro de volta"
7008,1,Produto totalmente diferente da foto do anúncio. Material de péssima qualidade. Aguardando contato da empresa para devolução. Até agora nada...
7597,1,Não recebi o produto e estou esperando a devolução do meu dinheiro.


In [168]:

# Step 8: Expand unresolved refund/return detection
# Reuses reviews_analysis and normalized_text.

additional_unresolved_terms = [
    "esperando o reembolso",
    "esperando reembolso",
    "aguardando o reembolso",
    "aguardando reembolso",
    "esperando o estorno",
    "aguardando o estorno",
    "ainda nao recebi o valor",
    "nao devolveram o dinheiro",
    "ate hoje nao recebi",
    "ate hoje nao devolveram",
    "ate hoje nada",
    "faz um mes e nada",
    "mais de um mes e nada",
    "estou no aguardo",
    "continuo aguardando",
    "nao obtive resposta",
    "nao tive resposta",
    "nao retornaram",
    "sem retorno",
    "sem qualquer retorno",
    "nao deram nenhuma resposta",
    "nao recebi resposta",
    "nao consigo fazer a devolucao"
]

additional_matches = (
    reviews_analysis["normalized_text"].apply(
        lambda text: contains_any(text, additional_unresolved_terms)
    )
    & reviews_analysis["refund_return_mentioned"]
)

previous_count = int(
    reviews_analysis["refund_return_problem_candidate"].sum()
)

reviews_analysis["refund_return_problem_candidate"] = (
    reviews_analysis["refund_return_problem_candidate"]
    | additional_matches
)

updated_count = int(
    reviews_analysis["refund_return_problem_candidate"].sum()
)

print(f"Previous candidates: {previous_count}")
print(f"Updated candidates: {updated_count}")
print(f"New candidates added: {updated_count - previous_count}")

display(
    reviews_analysis.loc[
        additional_matches
        & ~reviews_analysis["refund_return_problem_candidate"].shift(
            0, fill_value=False
        ),
        ["review_score", "combined_text"]
    ].head(20)
)


Previous candidates: 85
Updated candidates: 85
New candidates added: 0


,review_score,combined_text


In [169]:

# Step 9: Prepare review analysis outputs for Power BI
# Reuses reviews_analysis and theme_keywords.

from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Keep original text, review identifiers, scores, and theme flags.
theme_columns = list(theme_keywords.keys())

extra_columns = [
    "refund_return_mentioned",
    "refund_return_problem_candidate"
]

base_columns = [
    col for col in [
        "review_id", "order_id", "review_score",
        "review_comment_title", "review_comment_message",
        "combined_text", "has_text"
    ]
    if col in reviews_analysis.columns
]

export_columns = list(dict.fromkeys(
    base_columns
    + theme_columns
    + [col for col in extra_columns if col in reviews_analysis.columns]
))

review_export = reviews_analysis[export_columns].copy()

# Recalculate the theme summary from the current labels.
text_reviews = reviews_analysis.loc[
    reviews_analysis["has_text"]
].copy()

summary_rows = []

for theme in theme_columns:
    matched = text_reviews.loc[text_reviews[theme]]
    count = len(matched)

    summary_rows.append({
        "theme": theme,
        "review_count": count,
        "pct_of_text_reviews": round(
            count / len(text_reviews) * 100, 2
        ) if len(text_reviews) else 0,
        "average_score": round(
            matched["review_score"].mean(), 2
        ) if count else None,
        "pct_low_rating_1_or_2": round(
            matched["review_score"].le(2).mean() * 100, 2
        ) if count else None
    })

final_theme_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("review_count", ascending=False)
    .reset_index(drop=True)
)

return_summary = pd.DataFrame([{
    "reviews_with_text": len(text_reviews),
    "reviews_mentioning_return_refund": int(
        reviews_analysis["refund_return_mentioned"].sum()
    ),
    "potential_return_refund_problems": int(
        reviews_analysis["refund_return_problem_candidate"].sum()
    ),
    "classification_method": "Rule-based keyword matching",
    "validation_status": "Provisional; not manually validated in full"
}])

# Save outputs.
review_export.to_csv(
    output_dir / "review_theme_classification.csv",
    index=False,
    encoding="utf-8-sig"
)

final_theme_summary.to_csv(
    output_dir / "review_theme_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

return_summary.to_csv(
    output_dir / "review_return_refund_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Export complete.")
print("Review records exported:", len(review_export))
print("Text-bearing reviews:", int(review_export["has_text"].sum()))
print("Review-level columns:", len(review_export.columns))
print("Output folder:", output_dir.resolve())

display(final_theme_summary)
display(return_summary)


Export complete.
Review records exported: 99224
Text-bearing reviews: 42687
Review-level columns: 20
Output folder: C:\Users\PC-FMS\OneDrive\Documents\Projects\marketplace-intel\data\processed


,theme,review_count,pct_of_text_reviews,average_score,pct_low_rating_1_or_2
0,non_delivery,3054,7.15,1.43,86.90
1,late_delivery,833,1.95,2.99,43.22
2,product_defect_or_damage,639,1.50,1.79,76.53
3,missing_order_items,426,1.00,1.69,77.23
4,support_complaint,214,0.50,1.27,91.59
5,wrong_product,192,0.45,1.49,85.42
6,freight_or_shipping_cost,104,0.24,3.09,28.85
7,authenticity_concerns,97,0.23,1.48,83.51
8,product_description_mismatch,82,0.19,1.89,69.51
9,packaging_damage,21,0.05,2.52,47.62


,reviews_with_text,reviews_mentioning_return_refund,potential_return_refund_problems,classification_method,validation_status
0,42687,823,85,Rule-based keyword matching,Provisional; not manually validated in full


In [170]:

# Step 10: Synchronize the return/refund theme and refresh exports

# Update the dashboard theme flag from the latest candidate classification.
reviews_analysis["refund_or_return_problem"] = (
    reviews_analysis["refund_return_problem_candidate"].fillna(False).astype(bool)
)

# Rebuild the theme summary using the synchronized labels.
theme_columns = list(theme_keywords.keys())

text_reviews = reviews_analysis.loc[
    reviews_analysis["has_text"]
].copy()

summary_rows = []

for theme in theme_columns:
    matched = text_reviews.loc[text_reviews[theme]]
    count = len(matched)

    summary_rows.append({
        "theme": theme,
        "review_count": count,
        "pct_of_text_reviews": round(
            count / len(text_reviews) * 100, 2
        ) if len(text_reviews) else 0,
        "average_score": round(
            matched["review_score"].mean(), 2
        ) if count else None,
        "pct_low_rating_1_or_2": round(
            matched["review_score"].le(2).mean() * 100, 2
        ) if count else None
    })

final_theme_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("review_count", ascending=False)
    .reset_index(drop=True)
)

# Refresh the review-level export.
base_columns = [
    col for col in [
        "review_id", "order_id", "review_score",
        "review_comment_title", "review_comment_message",
        "combined_text", "has_text"
    ]
    if col in reviews_analysis.columns
]

extra_columns = [
    "refund_return_mentioned",
    "refund_return_problem_candidate"
]

export_columns = list(dict.fromkeys(
    base_columns
    + theme_columns
    + [c for c in extra_columns if c in reviews_analysis.columns]
))

review_export = reviews_analysis[export_columns].copy()

output_dir = Path("../data/processed")

review_export.to_csv(
    output_dir / "review_theme_classification.csv",
    index=False,
    encoding="utf-8-sig"
)

final_theme_summary.to_csv(
    output_dir / "review_theme_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

return_summary = pd.DataFrame([{
    "reviews_with_text": len(text_reviews),
    "reviews_mentioning_return_refund": int(
        reviews_analysis["refund_return_mentioned"].sum()
    ),
    "potential_return_refund_problems": int(
        reviews_analysis["refund_return_problem_candidate"].sum()
    ),
    "classification_method": "Rule-based keyword matching",
    "validation_status": "Provisional; not manually validated in full"
}])

return_summary.to_csv(
    output_dir / "review_return_refund_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Updated review export rows:", len(review_export))
print(
    "Return/refund problem labels:",
    int(review_export["refund_or_return_problem"].sum())
)
display(final_theme_summary)


Updated review export rows: 99224
Return/refund problem labels: 85


,theme,review_count,pct_of_text_reviews,average_score,pct_low_rating_1_or_2
0,non_delivery,3054,7.15,1.43,86.90
1,late_delivery,833,1.95,2.99,43.22
2,product_defect_or_damage,639,1.50,1.79,76.53
3,missing_order_items,426,1.00,1.69,77.23
4,support_complaint,214,0.50,1.27,91.59
5,wrong_product,192,0.45,1.49,85.42
6,freight_or_shipping_cost,104,0.24,3.09,28.85
7,authenticity_concerns,97,0.23,1.48,83.51
8,refund_or_return_problem,85,0.20,1.46,87.06
9,product_description_mismatch,82,0.19,1.89,69.51


In [1]:
import pandas as pd
from pathlib import Path

regional_delivery = pd.DataFrame({
    "brazil_region": [
        "Northeast",
        "North",
        "Central-West",
        "Southeast",
        "South"
    ],
    "delivered_orders": [
        9044, 1796, 5624, 66193, 13813
    ],
    "late_orders": [
        1296, 176, 448, 4932, 974
    ],
    "late_delivery_pct": [
        14.33, 9.80, 7.97, 7.45, 7.05
    ],
    "avg_days_vs_estimate": [
        -10.65, -14.93, -11.64, -10.86, -12.38
    ],
    "avg_days_late": [
        11.71, 12.47, 8.40, 9.23, 8.29
    ],
    "median_days_late": [
        7.67, 7.05, 5.61, 5.54, 5.76
    ]
})

regional_delivery

,brazil_region,delivered_orders,late_orders,late_delivery_pct,avg_days_vs_estimate,avg_days_late,median_days_late
0,Northeast,9044,1296,14.33,-10.65,11.71,7.67
1,North,1796,176,9.80,-14.93,12.47,7.05
2,Central-West,5624,448,7.97,-11.64,8.40,5.61
3,Southeast,66193,4932,7.45,-10.86,9.23,5.54
4,South,13813,974,7.05,-12.38,8.29,5.76


In [2]:
project_root = Path(
    r"C:\Users\PC-FMS\OneDrive\Documents\Projects\marketplace-intel"
)

output_path = (
    project_root
    / "data"
    / "processed"
    / "regional_delivery_summary.csv"
)

regional_delivery.to_csv(output_path, index=False)

print("Regional delivery CSV exported successfully!")
print(output_path)

Regional delivery CSV exported successfully!
C:\Users\PC-FMS\OneDrive\Documents\Projects\marketplace-intel\data\processed\regional_delivery_summary.csv


In [3]:
print("Shape:", regional_delivery.shape)
print("Missing values:", regional_delivery.isnull().sum().sum())
print("Duplicate rows:", regional_delivery.duplicated().sum())

Shape: (5, 7)
Missing values: 0
Duplicate rows: 0


In [4]:
delivery_stage_summary = pd.DataFrame({
    "delivery_stage": [
        "Approval to carrier",
        "Carrier to customer"
    ],
    "valid_orders": [
        95112,
        96446
    ],
    "average_days": [
        2.85,
        9.33
    ],
    "median_days": [
        1.85,
        7.10
    ]
})

delivery_stage_summary

,delivery_stage,valid_orders,average_days,median_days
0,Approval to carrier,95112,2.85,1.85
1,Carrier to customer,96446,9.33,7.10


In [5]:
output_path = (
    project_root
    / "data"
    / "processed"
    / "delivery_stage_summary.csv"
)

delivery_stage_summary.to_csv(output_path, index=False)

print("Exported:", output_path)

Exported: C:\Users\PC-FMS\OneDrive\Documents\Projects\marketplace-intel\data\processed\delivery_stage_summary.csv


In [6]:
print(delivery_stage_summary.shape)
print(delivery_stage_summary.isnull().sum())

(2, 4)
delivery_stage    0
valid_orders      0
average_days      0
median_days       0
dtype: int64


In [1]:
import pandas as pd

monthly_customer_data = [
    ("2017-01", 755, 754, 1, 0.13),
    ("2017-02", 1708, 1705, 3, 0.18),
    ("2017-03", 2601, 2595, 6, 0.23),
    ("2017-04", 2359, 2339, 20, 0.85),
    ("2017-05", 3588, 3560, 28, 0.78),
    ("2017-06", 3154, 3114, 40, 1.27),
    ("2017-07", 3894, 3843, 51, 1.31),
    ("2017-08", 4211, 4149, 62, 1.47),
    ("2017-09", 4170, 4090, 80, 1.92),
    ("2017-10", 4501, 4412, 89, 1.98),
    ("2017-11", 7342, 7217, 125, 1.70),
    ("2017-12", 5557, 5442, 115, 2.07),
    ("2018-01", 7120, 6983, 137, 1.92),
    ("2018-02", 6537, 6422, 115, 1.76),
    ("2018-03", 7096, 6947, 149, 2.10),
    ("2018-04", 6878, 6709, 169, 2.46),
    ("2018-05", 6795, 6604, 191, 2.81),
    ("2018-06", 6121, 5934, 187, 3.06),
    ("2018-07", 6211, 6054, 157, 2.53),
    ("2018-08", 6411, 6238, 173, 2.70)
]

monthly_customer_summary = pd.DataFrame(
    monthly_customer_data,
    columns=[
        "order_month",
        "active_customers",
        "new_customers",
        "returning_customers",
        "returning_customer_pct"
    ]
)

monthly_customer_summary.head()

,order_month,active_customers,new_customers,returning_customers,returning_customer_pct
0,2017-01,755,754,1,0.13
1,2017-02,1708,1705,3,0.18
2,2017-03,2601,2595,6,0.23
3,2017-04,2359,2339,20,0.85
4,2017-05,3588,3560,28,0.78


In [2]:
output_path = (
    project_root
    / "data"
    / "processed"
    / "monthly_customer_summary.csv"
)

monthly_customer_summary.to_csv(output_path, index=False)

print("Exported:", output_path)

NameError: name 'project_root' is not defined

In [3]:
from pathlib import Path

project_root = Path(
    r"C:\Users\PC-FMS\OneDrive\Documents\Projects\marketplace-intel"
)

output_path = (
    project_root
    / "data"
    / "processed"
    / "monthly_customer_summary.csv"
)

print("Output path:", output_path)

Output path: C:\Users\PC-FMS\OneDrive\Documents\Projects\marketplace-intel\data\processed\monthly_customer_summary.csv


In [4]:
monthly_customer_summary.to_csv(output_path, index=False)

print("Export successful:", output_path.exists())

Export successful: True


In [5]:
pd.read_csv(output_path).head()

,order_month,active_customers,new_customers,returning_customers,returning_customer_pct
0,2017-01,755,754,1,0.13
1,2017-02,1708,1705,3,0.18
2,2017-03,2601,2595,6,0.23
3,2017-04,2359,2339,20,0.85
4,2017-05,3588,3560,28,0.78
